In [1]:
# ============================================
# CELDA 1: Imports y configuracion inicial
# ============================================
import micropip
await micropip.install('ipywidgets')

import sqlite3
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output
import base64
from datetime import datetime
import re

print("Sistema cargado correctamente")

Sistema cargado correctamente


In [2]:
# ============================================
# CELDA 2: Base de datos en memoria (CORREGIDA)
# ============================================
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

cursor.executescript("""
CREATE TABLE categorias (
    id_categoria INTEGER PRIMARY KEY,
    nombre TEXT NOT NULL UNIQUE,
    descripcion TEXT,
    id_padre INTEGER,
    nivel INTEGER DEFAULT 0,
    activo INTEGER DEFAULT 1,
    FOREIGN KEY (id_padre) REFERENCES categorias(id_categoria)
);

CREATE TABLE proveedores (
    id_proveedor INTEGER PRIMARY KEY,
    razon_social TEXT NOT NULL,
    nombre_comercial TEXT,
    rfc TEXT UNIQUE,
    direccion TEXT,
    telefono TEXT,
    email TEXT,
    contacto_nombre TEXT,
    dias_credito INTEGER DEFAULT 0,
    activo INTEGER DEFAULT 1,
    fecha_registro TEXT DEFAULT CURRENT_TIMESTAMP
);

CREATE TABLE clientes (
    id_cliente INTEGER PRIMARY KEY,
    tipo_cliente TEXT CHECK(tipo_cliente IN ('persona', 'empresa')),
    nombre TEXT NOT NULL,
    apellidos TEXT,
    rfc TEXT UNIQUE,
    email TEXT,
    telefono TEXT,
    direccion TEXT,
    direccion_envio TEXT,
    limite_credito REAL DEFAULT 0,
    saldo_pendiente REAL DEFAULT 0,
    activo INTEGER DEFAULT 1,
    fecha_registro TEXT DEFAULT CURRENT_TIMESTAMP
);

CREATE TABLE usuarios (
    id_usuario INTEGER PRIMARY KEY,
    username TEXT NOT NULL UNIQUE,
    password_hash TEXT NOT NULL,
    nombre_completo TEXT NOT NULL,
    email TEXT NOT NULL UNIQUE,
    rol TEXT CHECK(rol IN ('admin', 'gerente', 'vendedor', 'almacenista', 'auditor')),
    activo INTEGER DEFAULT 1,
    ultimo_acceso TEXT,
    intentos_fallidos INTEGER DEFAULT 0,
    fecha_registro TEXT DEFAULT CURRENT_TIMESTAMP
);

CREATE TABLE productos (
    id_producto INTEGER PRIMARY KEY,
    sku TEXT NOT NULL UNIQUE,
    codigo_barras TEXT,
    nombre TEXT NOT NULL,
    descripcion TEXT,
    id_categoria INTEGER,
    id_proveedor INTEGER,
    precio_costo REAL DEFAULT 0,
    precio_venta REAL DEFAULT 0,
    stock_minimo INTEGER DEFAULT 0,
    stock_maximo INTEGER DEFAULT 0,
    stock_actual INTEGER DEFAULT 0,
    unidad_medida TEXT DEFAULT 'pieza',
    ubicacion_almacen TEXT,
    activo INTEGER DEFAULT 1,
    fecha_registro TEXT DEFAULT CURRENT_TIMESTAMP,
    fecha_actualizacion TEXT DEFAULT CURRENT_TIMESTAMP,
    FOREIGN KEY (id_categoria) REFERENCES categorias(id_categoria),
    FOREIGN KEY (id_proveedor) REFERENCES proveedores(id_proveedor)
);

CREATE TABLE movimientos (
    id_movimiento INTEGER PRIMARY KEY,
    tipo TEXT CHECK(tipo IN ('entrada', 'salida')),
    id_producto INTEGER NOT NULL,
    id_cliente INTEGER,
    id_proveedor INTEGER,
    cantidad INTEGER NOT NULL,
    precio_unitario REAL,
    total REAL,
    stock_anterior INTEGER,
    stock_posterior INTEGER,
    referencia_documento TEXT,
    motivo TEXT,
    id_usuario INTEGER NOT NULL,
    activo INTEGER DEFAULT 1,
    fecha_movimiento TEXT DEFAULT CURRENT_TIMESTAMP,
    FOREIGN KEY (id_producto) REFERENCES productos(id_producto),
    FOREIGN KEY (id_cliente) REFERENCES clientes(id_cliente),
    FOREIGN KEY (id_proveedor) REFERENCES proveedores(id_proveedor),
    FOREIGN KEY (id_usuario) REFERENCES usuarios(id_usuario)
);
""")

# Datos de prueba - CATEGORIAS EXPANDIDAS TIPO PALACIO DE HIERRO
# IDs 1-6: ORIGINALES (sin cambio para compatibilidad)
# IDs 7-42: NUEVAS (consecutivos, sin duplicados)

cursor.executemany("INSERT INTO categorias (id_categoria, nombre, descripcion, id_padre, nivel) VALUES (?, ?, ?, ?, ?)",
    [
    # ============================================
    # NIVEL 0 - Departamentos principales (Raices)
    # ============================================
    (1, 'Electronica', 'Dispositivos electronicos', None, 0),
    (4, 'Alimentos', 'Productos comestibles', None, 0),
    (5, 'Textil', 'Ropa y accesorios', None, 0),
    (7, 'Hogar', 'Articulos para el hogar', None, 0),
    (8, 'Belleza', 'Cosmeticos y cuidado personal', None, 0),
    (9, 'Deportes', 'Articulos deportivos', None, 0),
    (10, 'Juguetes', 'Juguetes y juegos', None, 0),
    (11, 'Muebles', 'Muebles para hogar y oficina', None, 0),
    (12, 'Relojes y Joyeria', 'Relojes, joyas y accesorios', None, 0),
    (13, 'Zapatos', 'Calzado para toda la familia', None, 0),
    
    # ============================================
    # NIVEL 1 - Subcategorias de Electronica (id_padre = 1)
    # ============================================
    (2, 'Computadoras', 'PCs y laptops', 1, 1),
    (3, 'Smartphones', 'Telefonos inteligentes', 1, 1),
    (14, 'Televisores', 'TVs y pantallas', 1, 1),
    (15, 'Audio', 'Equipos de sonido y audifonos', 1, 1),
    (16, 'Camaras', 'Camaras fotograficas y de video', 1, 1),
    (17, 'Videojuegos', 'Consolas y videojuegos', 1, 1),
    
    # ============================================
    # NIVEL 1 - Subcategorias de Alimentos (id_padre = 4)
    # ============================================
    (6, 'Bebidas', 'Bebidas', 4, 1),
    (18, 'Despensa', 'Productos basicos de cocina', 4, 1),
    (19, 'Lacteos', 'Leche, queso y derivados', 4, 1),
    (20, 'Carnes', 'Carnes frias y frescas', 4, 1),
    (21, 'Panaderia', 'Pan y reposteria', 4, 1),
    
    # ============================================
    # NIVEL 1 - Subcategorias de Textil (id_padre = 5)
    # ============================================
    (22, 'Ropa Mujer', 'Vestidos, blusas, faldas', 5, 1),
    (23, 'Ropa Hombre', 'Camisas, pantalones, trajes', 5, 1),
    (24, 'Ropa Infantil', 'Ropa para ninos y bebes', 5, 1),
    (25, 'Ropa Deportiva', 'Ropa para ejercicio', 5, 1),
    (26, 'Lenceria', 'Ropa interior y pijamas', 5, 1),
    
    # ============================================
    # NIVEL 1 - Subcategorias de Hogar (id_padre = 7)
    # ============================================
    (27, 'Cocina', 'Utensilios y electrodomesticos de cocina', 7, 1),
    (28, 'Bano', 'Articulos para bano', 7, 1),
    (29, 'Decoracion', 'Adornos y decoracion', 7, 1),
    (30, 'Cama', 'Ropa de cama y colchas', 7, 1),
    (31, 'Mesa', 'Vajillas y cristaleria', 7, 1),
    
    # ============================================
    # NIVEL 1 - Subcategorias de Belleza (id_padre = 8)
    # ============================================
    (32, 'Perfumes', 'Fragancias para hombre y mujer', 8, 1),
    (33, 'Maquillaje', 'Cosmeticos y productos de belleza', 8, 1),
    (34, 'Cuidado de la Piel', 'Cremas y tratamientos faciales', 8, 1),
    (35, 'Cuidado del Cabello', 'Shampoo, acondicionador y tratamientos', 8, 1),
    
    # ============================================
    # NIVEL 1 - Subcategorias de Deportes (id_padre = 9)
    # ============================================
    (36, 'Gimnasio', 'Equipos para ejercicio en casa', 9, 1),
    (37, 'Ciclismo', 'Bicicletas y accesorios', 9, 1),
    (38, 'Running', 'Zapatos y ropa para correr', 9, 1),
    (39, 'Natacion', 'Trajes de bano y accesorios', 9, 1),
    
    # ============================================
    # NIVEL 1 - Subcategorias de Juguetes (id_padre = 10)
    # ============================================
    (40, 'Juguetes Ninos', 'Juguetes para ninos 3-12 anos', 10, 1),
    (41, 'Juguetes Bebe', 'Juguetes para bebes 0-3 anos', 10, 1),
    (42, 'Juegos de Mesa', 'Juegos de mesa familiares', 10, 1),
    (43, 'Munecas', 'Munecas y accesorios', 10, 1),
    
    # ============================================
    # NIVEL 1 - Subcategorias de Muebles (id_padre = 11)
    # ============================================
    (44, 'Sala', 'Sofas, sillones y mesas de centro', 11, 1),
    (45, 'Comedor', 'Mesas y sillas de comedor', 11, 1),
    (46, 'Recamara', 'Camas, buros y comodas', 11, 1),
    (47, 'Oficina', 'Escritorios y sillas de oficina', 11, 1),
    
    # ============================================
    # NIVEL 1 - Subcategorias de Relojes y Joyeria (id_padre = 12)
    # ============================================
    (48, 'Relojes Hombre', 'Relojes para caballero', 12, 1),
    (49, 'Relojes Mujer', 'Relojes para dama', 12, 1),
    (50, 'Joyeria Fina', 'Oro, plata y piedras preciosas', 12, 1),
    (51, 'Bisuteria', 'Accesorios de moda', 12, 1),
    
    # ============================================
    # NIVEL 1 - Subcategorias de Zapatos (id_padre = 13)
    # ============================================
    (52, 'Zapatos Mujer', 'Zapatos, botas y sandalias dama', 13, 1),
    (53, 'Zapatos Hombre', 'Zapatos, botines y tenis caballero', 13, 1),
    (54, 'Zapatos Ninos', 'Calzado infantil', 13, 1),
    ])

cursor.executemany("INSERT INTO proveedores (id_proveedor, razon_social, nombre_comercial, rfc, telefono, email, contacto_nombre, dias_credito) VALUES (?, ?, ?, ?, ?, ?, ?, ?)",
    [(1, 'TechSupply S.A.', 'TechSupply', 'TSC123456ABC', '5550101234', 'ventas@tech.com', 'Juan Perez', 30),
     (2, 'Alimentos del Norte', 'ADNorte', 'ADN987654XYZ', '5550202567', 'pedidos@adn.com', 'Maria Lopez', 15),
     (3, 'Textiles Mexicanos', 'TexMex', 'TXM456789DEF', '5550303890', 'contacto@txm.com', 'Carlos Ruiz', 0)])

cursor.executemany("INSERT INTO clientes (id_cliente, tipo_cliente, nombre, apellidos, rfc, email, telefono, direccion, limite_credito) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)",
    [(1, 'persona', 'Juan', 'Perez Garcia', 'PEGJ800101HDF', 'juan@email.com', '5551001456', 'Calle 1', 5000),
     (2, 'empresa', 'Soluciones Tecnologicas del Sur', None, 'STS900202ABC', 'sts@empresa.com', '5552002789', 'Av. Empresarial 100', 50000),
     (3, 'persona', 'Maria', 'Lopez Santos', 'LOSM850303XYZ', 'maria@email.com', '5551003234', 'Calle 3', 10000)])

cursor.executemany("INSERT INTO usuarios (id_usuario, username, password_hash, nombre_completo, email, rol) VALUES (?, ?, ?, ?, ?, ?)",
    [(1, 'admin', '[HASH_SEGURO]', 'Administrador del Sistema', 'admin@empresa.com', 'admin'),
     (2, 'jperez', '[HASH_SEGURO]', 'Juan Perez Lopez', 'jperez@empresa.com', 'vendedor'),
     (3, 'mgarcia', '[HASH_SEGURO]', 'Maria Garcia Ruiz', 'mgarcia@empresa.com', 'almacenista')])

cursor.executemany("INSERT INTO productos (id_producto, sku, nombre, descripcion, id_categoria, id_proveedor, precio_costo, precio_venta, stock_actual, stock_minimo, ubicacion_almacen) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)",
    [(1, 'LAP-001', 'Laptop Pro 15"', 'Laptop alta performance', 2, 1, 8500, 12000, 25, 5, 'A-01-05'),
     (2, 'TEL-001', 'Smartphone X 128GB', 'Telefono ultima generacion', 3, 1, 5000, 8000, 15, 3, 'A-02-03'),
     (3, 'ARR-001', 'Arroz Integral 1kg', 'Arroz de grano largo', 6, 2, 18.50, 28, 150, 20, 'B-01-01'),
     (4, 'CAM-001', 'Camiseta Algodon M', 'Camiseta basica manga corta', 5, 3, 85, 199, 50, 10, 'C-01-01'),
     (5, 'LAP-002', 'Laptop Gamer 17"', 'Laptop gaming RTX 4060', 2, 1, 15000, 22000, 8, 2, 'A-01-06')])

# Datos de prueba para movimientos
cursor.executemany("INSERT INTO movimientos (id_movimiento, tipo, id_producto, id_cliente, id_proveedor, cantidad, precio_unitario, total, stock_anterior, stock_posterior, id_usuario, referencia_documento, motivo, activo) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)",
    [(1, 'entrada', 1, None, 1, 50, 8500, 425000, 0, 50, 1, 'FAC-001', 'Compra inicial', 1),
     (2, 'entrada', 2, None, 1, 30, 5000, 150000, 0, 30, 1, 'FAC-002', 'Compra inicial', 1),
     (3, 'entrada', 3, None, 2, 200, 18.50, 3700, 0, 200, 1, 'FAC-003', 'Compra inicial', 1),
     (4, 'salida', 1, 1, None, 5, 12000, 60000, 50, 45, 2, 'REM-001', 'Venta a cliente', 1),
     (5, 'salida', 2, 2, None, 3, 8000, 24000, 30, 27, 2, 'REM-002', 'Venta empresa', 1)])

conn.commit()
print("Base de datos creada con datos de prueba ampliados - IDs corregidos y consecutivos")

Base de datos creada con datos de prueba ampliados - IDs corregidos y consecutivos


In [3]:
# ============================================
# CELDA 3: Funciones utilitarias y Validadores (CORREGIDA)
# ============================================

def mostrar_mensaje(tipo, texto, output_widget):
    """Muestra mensaje en el output especifico"""
    with output_widget:
        clear_output()
        colores = {
            'exito': ('#d4edda', '#155724'),
            'error': ('#f8d7da', '#721c24'),
            'info': ('#d1ecf1', '#0c5460'),
            'advertencia': ('#fff3cd', '#856404')
        }
        bg, fg = colores.get(tipo, ('#e2e3e5', '#383d41'))
        display(widgets.HTML(
            "<div style='padding:10px; background:{}; color:{}; "
            "border-radius:5px; margin:5px 0;'><b>{}:</b> {}</div>".format(
                bg, fg, tipo.upper(), texto)
        ))

def crear_enlace_csv(df, nombre_archivo):
    csv = df.to_csv(index=False)
    b64 = base64.b64encode(csv.encode()).decode()
    return '<a href="data:text/csv;base64,{}" download="{}" ' \
           'style="padding:8px 15px; background:#28a745; color:white; ' \
           'text-decoration:none; border-radius:4px; font-size:12px;">Descargar {}</a>'.format(
               b64, nombre_archivo, nombre_archivo)

# ============================================
# SISTEMA DE VALIDACION (SIN RFC, EMAIL, TELEFONO)
# ============================================

class Validador:
    """Clase para validar datos antes de enviar a SQLite"""
    
    MENSAJES = {
        'requerido': 'Este campo es obligatorio',
        'numero': 'Debe ser un numero valido',
        'unico': 'Este ID ya existe en la base de datos',
        'positivo': 'Debe ser un valor positivo',
        'rango': 'Valor fuera del rango permitido'
    }
    
    @classmethod
    def validar(cls, valor, tipo, requerido=False, min_val=None, max_val=None, 
                tabla=None, campo=None, id_actual=None):
        if requerido and (valor is None or str(valor).strip() == ''):
            return False, cls.MENSAJES['requerido']
        
        if not requerido and (valor is None or str(valor).strip() == ''):
            return True, None
        
        valor_str = str(valor).strip()
        
        if tipo in ['numero', 'slider']:
            try:
                num = int(valor_str)
                if min_val is not None and num < min_val:
                    return False, "{} (min: {})".format(cls.MENSAJES['rango'], min_val)
                if max_val is not None and num > max_val:
                    return False, "{} (max: {})".format(cls.MENSAJES['rango'], max_val)
                if num < 0:
                    return False, cls.MENSAJES['positivo']
            except ValueError:
                return False, cls.MENSAJES['numero']
                
        elif tipo in ['decimal', 'float_slider']:
            try:
                num = float(valor_str)
                if min_val is not None and num < min_val:
                    return False, "{} (min: {})".format(cls.MENSAJES['rango'], min_val)
                if max_val is not None and num > max_val:
                    return False, "{} (max: {})".format(cls.MENSAJES['rango'], max_val)
                if num < 0:
                    return False, cls.MENSAJES['positivo']
            except ValueError:
                return False, "Debe ser un numero decimal valido"
        
        if tabla and campo and tipo in ['numero', 'slider']:
            try:
                cursor.execute("SELECT {} FROM {} WHERE {} = ?".format(campo, tabla, campo), (int(valor_str),))
                existe = cursor.fetchone()
                if existe and (id_actual is None or int(valor_str) != id_actual):
                    return False, cls.MENSAJES['unico']
            except:
                pass
        
        return True, None

def get_siguiente_id(tabla, campo_id):
    """Obtiene el siguiente ID disponible (max + 1)"""
    try:
        cursor.execute("SELECT MAX({}) FROM {}".format(campo_id, tabla))
        max_id = cursor.fetchone()[0]
        return (max_id or 0) + 1
    except:
        return 1

def get_categorias():
    """Obtiene TODAS las categorias activas para dropdowns generales"""
    try:
        cursor.execute("SELECT id_categoria, nombre FROM categorias WHERE activo=1 ORDER BY nombre")
        return [("{} (ID:{})".format(row[1], row[0]), row[0]) for row in cursor.fetchall()]
    except:
        return []

def get_categorias_padre():
    """Obtiene TODAS las categorias activas para seleccionar como padre (incluye todas las opciones)"""
    try:
        cursor.execute("""
            SELECT c.id_categoria, c.nombre, c.nivel, p.nombre as nombre_padre
            FROM categorias c
            LEFT JOIN categorias p ON c.id_padre = p.id_categoria
            WHERE c.activo = 1 
            ORDER BY c.nivel, c.nombre
        """)
        resultado = []
        for row in cursor.fetchall():
            id_cat, nombre, nivel, nombre_padre = row
            # Mostrar jerarquía visual: nombre (ID:id) [Nivel X]
            if nombre_padre:
                display_text = "{} → {} (ID:{}) [Niv.{}]".format(nombre_padre, nombre, id_cat, nivel)
            else:
                display_text = "{} (ID:{}) [Niv.{} - Raiz]".format(nombre, id_cat, nivel)
            resultado.append((display_text, id_cat))
        return [("Ninguna (raiz)", None)] + resultado
    except Exception as e:
        print("Error en get_categorias_padre:", str(e))
        return [("Ninguna (raiz)", None)]

def get_proveedores():
    try:
        cursor.execute("SELECT id_proveedor, razon_social FROM proveedores WHERE activo=1 ORDER BY razon_social")
        return [("{} (ID:{})".format(row[1], row[0]), row[0]) for row in cursor.fetchall()]
    except:
        return []

def get_clientes():
    try:
        cursor.execute("SELECT id_cliente, nombre, apellidos, tipo_cliente FROM clientes WHERE activo=1 ORDER BY nombre")
        return [("{} {} ({}) [ID:{}]".format(row[1], row[2] or '', row[3], row[0]), row[0]) for row in cursor.fetchall()]
    except:
        return []

def get_usuarios():
    try:
        cursor.execute("SELECT id_usuario, username, nombre_completo, rol FROM usuarios WHERE activo=1 ORDER BY username")
        return [("{} - {} ({}) [ID:{}]".format(row[1], row[2], row[3], row[0]), row[0]) for row in cursor.fetchall()]
    except:
        return []

def get_productos():
    try:
        cursor.execute("SELECT id_producto, sku, nombre, stock_actual FROM productos WHERE activo=1 ORDER BY nombre")
        return [("{}: {} (Stock:{}) [ID:{}]".format(row[1], row[2], row[3], row[0]), row[0]) for row in cursor.fetchall()]
    except:
        return []

print("Funciones utilitarias cargadas - get_categorias_padre() ahora muestra TODAS las categorias con jerarquia")

Funciones utilitarias cargadas - get_categorias_padre() ahora muestra TODAS las categorias con jerarquia


In [4]:
# ============================================
# CELDA 4: Clase CRUDTabla (Version con Callback de Refresco) - CORREGIDA
# ============================================

class CRUDTabla:
    def __init__(self, nombre_tabla, campos, on_operacion_completada=None):
        # Asignar TODOS los atributos primero antes de cualquier llamada a metodos
        self.nombre_tabla = nombre_tabla
        self.campos = campos
        self.modo_actual = 'insertar'
        self.id_seleccionado = None
        self.on_operacion_completada = on_operacion_completada  # Callback para notificar a otras tablas
        
        # Outputs independientes por instancia
        self.output_mensajes = widgets.Output()
        self.output_tabla = widgets.Output()
        self.output_confirmacion = widgets.Output()
        
        # Crear controles sin asignar callbacks todavia
        self._crear_controles_ui(nombre_tabla)
        self.crear_formulario()
        self.crear_layout()
        
        # Asignar callbacks despues de que todos los metodos estan definidos
        self._asignar_callbacks()
        
        # Mostrar tabla al inicio
        self.mostrar_tabla_completa()
    
    def _crear_controles_ui(self, nombre_tabla):
        """Crear la interfaz de controles sin callbacks"""
        # Radio buttons para modo
        self.radio_accion = widgets.RadioButtons(
            options=[
                ('[+] Insertar', 'insertar'),
                ('[*] Modificar', 'modificar'),
                ('[-] Borrar', 'borrar'),
                ('[?] Consultar', 'consultar')
            ],
            value='insertar',
            layout=widgets.Layout(width='100%', display='flex', flex_flow='row wrap')
        )
        
        # Botones (sin callbacks todavia)
        self.btn_ejecutar = widgets.Button(
            description='[Guardar]',
            button_style='success',
            layout=widgets.Layout(width='150px')
        )
        
        self.btn_limpiar = widgets.Button(
            description='[Limpiar]',
            button_style='warning',
            layout=widgets.Layout(width='150px')
        )
        
        self.btn_exportar = widgets.Button(
            description='[Exportar CSV]',
            button_style='info',
            layout=widgets.Layout(width='150px')
        )
        
        self.btn_cargar = widgets.Button(
            description='[Cargar Datos]',
            button_style='primary',
            layout=widgets.Layout(width='150px'),
            tooltip='Cargar datos del ID ingresado (solo Modificar/Borrar)'
        )
        
        # Usar el nombre de tabla pasado como parametro
        self.area_controles = widgets.VBox([
            widgets.HTML("<h3 style='color:#2c3e50; margin-top:0;'>{}</h3>".format(nombre_tabla.upper())),
            widgets.HBox([self.radio_accion], layout=widgets.Layout(margin='10px 0')),
            widgets.HBox([self.btn_ejecutar, self.btn_limpiar, self.btn_exportar, self.btn_cargar],
                        layout=widgets.Layout(margin='10px 0', gap='10px'))
        ])
    
    def _asignar_callbacks(self):
        """Asignar callbacks a los botones - llamado despues de que __init__ completa la definicion de metodos"""
        self.radio_accion.observe(self.on_cambio_accion, 'value')
        self.btn_ejecutar.on_click(self.on_ejecutar)
        self.btn_limpiar.on_click(self.on_limpiar)
        self.btn_exportar.on_click(self.on_exportar)
        self.btn_cargar.on_click(self.on_cargar_datos)
    
    def crear_formulario(self):
        self.widgets_campos = {}
        campos_widgets = []
        
        for campo in self.campos:
            w = self.crear_widget_campo(campo)
            self.widgets_campos[campo['nombre']] = w
            campos_widgets.append(w)
        
        self.area_formulario = widgets.VBox(
            campos_widgets,
            layout=widgets.Layout(margin='10px 0', padding='15px', 
                                border='1px solid #ddd', border_radius='5px')
        )
    
    def crear_widget_campo(self, campo):
        nombre = campo['nombre']
        tipo = campo['tipo']
        etiqueta = campo.get('etiqueta', nombre)
        default = campo.get('default', '')
        ayuda = campo.get('ayuda', '')
        
        layout_std = widgets.Layout(width='500px')
        layout_num = widgets.Layout(width='300px')
        
        tooltip = ayuda if ayuda else None
        
        if tipo == 'texto':
            return widgets.Text(description='{}:'.format(etiqueta), value=str(default), 
                              layout=layout_std, tooltip=tooltip)
        elif tipo == 'area':
            return widgets.Textarea(description='{}:'.format(etiqueta), value=str(default),
                                   layout=widgets.Layout(width='500px', height='60px'),
                                   tooltip=tooltip)
        elif tipo == 'numero':
            return widgets.IntText(description='{}:'.format(etiqueta), 
                                  value=default if default else 0,
                                  layout=layout_num, tooltip=tooltip)
        elif tipo == 'decimal':
            return widgets.FloatText(description='{}:'.format(etiqueta),
                                    value=default if default else 0.0,
                                    layout=layout_num, tooltip=tooltip)
        elif tipo == 'slider':
            return widgets.IntSlider(description='{}:'.format(etiqueta),
                                    min=campo.get('min', 0), max=campo.get('max', 100),
                                    value=default if default else 0,
                                    layout=layout_std, tooltip=tooltip)
        elif tipo == 'float_slider':
            return widgets.FloatSlider(description='{}:'.format(etiqueta),
                                      min=campo.get('min', 0), max=campo.get('max', 100),
                                      step=campo.get('step', 0.01),
                                      value=default if default else 0.0,
                                      layout=layout_std, tooltip=tooltip)
        elif tipo == 'dropdown':
            return widgets.Dropdown(description='{}:'.format(etiqueta),
                                   options=campo.get('opciones', []),
                                   value=default, layout=layout_std, tooltip=tooltip)
        elif tipo == 'radio':
            return widgets.RadioButtons(description='{}:'.format(etiqueta),
                                       options=campo.get('opciones', []),
                                       value=default, layout=layout_std, tooltip=tooltip)
        elif tipo == 'checkbox':
            return widgets.Checkbox(description=etiqueta, value=bool(default),
                                   layout=layout_num, tooltip=tooltip)
        elif tipo == 'fecha':
            return widgets.Text(description='{}:'.format(etiqueta), 
                               value=str(datetime.now())[:19], disabled=True, layout=layout_num)
        else:
            return widgets.Text(description='{}:'.format(etiqueta), value='', disabled=True)
    
    def crear_layout(self):
        self.vista = widgets.VBox([
            self.area_controles,
            self.area_formulario,
            self.output_mensajes,
            self.output_confirmacion,
            self.output_tabla
        ], layout=widgets.Layout(padding='10px'))
    
    def on_cambio_accion(self, change):
        self.modo_actual = change['new']
        self.id_seleccionado = None
        
        # Limpiar confirmacion
        with self.output_confirmacion:
            clear_output()
        
        textos = {
            'insertar': '[Guardar]',
            'modificar': '[Actualizar]',
            'borrar': '[Confirmar Borrado]',
            'consultar': '[Buscar]'
        }
        self.btn_ejecutar.description = textos.get(self.modo_actual, 'Ejecutar')
        self.btn_ejecutar.button_style = 'danger' if self.modo_actual == 'borrar' else 'success'
        
        # Configurar visibilidad de boton Cargar
        self.btn_cargar.layout.display = 'flex' if self.modo_actual in ['modificar', 'borrar'] else 'none'
        
        self.configurar_campos_por_modo()
        
        # Siempre mostrar tabla completa, excepto en consultar que se muestra despues de configurar campos
        if self.modo_actual != 'consultar':
            self.mostrar_tabla_completa()
    
    def configurar_campos_por_modo(self):
        """Configura campos segun el modo de operacion"""
        for campo in self.campos:
            nombre = campo['nombre']
            widget = self.widgets_campos[nombre]
            es_id = campo.get('es_id', False)
            es_editable = campo.get('editable', True)
            
            if self.modo_actual == 'insertar':
                # ID sugerido pero editable, otros campos editables
                if es_id:
                    widget.disabled = False
                    widget.value = get_siguiente_id(self.nombre_tabla, nombre)
                else:
                    widget.disabled = not es_editable
                    self.resetear_valor(campo, widget)
                    
            elif self.modo_actual == 'modificar':
                # ID editable para buscar, luego readonly cuando se cargan datos
                if es_id:
                    widget.disabled = False
                    widget.value = 0
                else:
                    widget.disabled = True
                    self.resetear_valor(campo, widget)
                    
            elif self.modo_actual == 'borrar':
                # ID editable para buscar, otros deshabilitados
                if es_id:
                    widget.disabled = False
                    widget.value = 0
                else:
                    widget.disabled = True
                    self.resetear_valor(campo, widget)
                    
            elif self.modo_actual == 'consultar':
                # Todos editables para criterios de busqueda, valores vacios
                widget.disabled = False
                self.limpiar_campo_consulta(campo, widget)
        
        # Mostrar tabla completa en modo consultar tambien
        if self.modo_actual == 'consultar':
            self.mostrar_tabla_completa()
    
    def limpiar_campo_consulta(self, campo, widget):
        """Limpia completamente un campo para modo consulta"""
        tipo = campo['tipo']
        
        try:
            if tipo in ['texto', 'area', 'fecha']:
                widget.value = ''
            elif tipo in ['numero', 'decimal']:
                widget.value = 0
            elif tipo == 'slider':
                min_val = campo.get('min', 0)
                widget.value = min_val
            elif tipo == 'float_slider':
                min_val = campo.get('min', 0.0)
                widget.value = min_val
            elif tipo == 'checkbox':
                widget.value = False
            elif tipo == 'dropdown':
                try:
                    widget.value = None
                except:
                    pass
            elif tipo == 'radio':
                try:
                    widget.value = None
                except:
                    pass
        except Exception as e:
            pass
    
    def resetear_valor(self, campo, widget):
        default = campo.get('default', '')
        tipo = campo['tipo']
        
        try:
            if tipo in ['texto', 'area']:
                widget.value = str(default) if default else ''
            elif tipo in ['numero', 'slider']:
                widget.value = default if default else 0
            elif tipo in ['decimal', 'float_slider']:
                widget.value = default if default else 0.0
            elif tipo == 'checkbox':
                widget.value = bool(default)
            elif tipo == 'dropdown':
                widget.value = default if default else (widget.options[0][1] if widget.options else None)
        except:
            pass
    
    def on_cargar_datos(self, b):
        """Carga datos del ID ingresado (para Modificar y Borrar)"""
        if self.modo_actual not in ['modificar', 'borrar']:
            return
            
        campo_id = next((c for c in self.campos if c.get('es_id', False)), None)
        if not campo_id:
            return
            
        id_valor = self.widgets_campos[campo_id['nombre']].value
        
        if not id_valor or id_valor == 0:
            mostrar_mensaje('error', 'Ingrese un ID valido para cargar', self.output_mensajes)
            return
        
        try:
            cursor.execute("SELECT * FROM {} WHERE {} = ? AND activo = 1".format(
                self.nombre_tabla, campo_id['nombre']), (id_valor,))
            registro = cursor.fetchone()
            
            if not registro:
                mostrar_mensaje('error', 'No existe registro con ID {} o esta inactivo'.format(id_valor), self.output_mensajes)
                return
            
            columnas = [c['nombre'] for c in self.campos]
            for i, campo in enumerate(self.campos):
                nombre = campo['nombre']
                valor = registro[i]
                widget = self.widgets_campos[nombre]
                
                if valor is not None:
                    widget.value = valor
                else:
                    self.resetear_valor(campo, widget)
                
                if self.modo_actual == 'modificar':
                    if campo.get('es_id', False):
                        widget.disabled = True
                        self.id_seleccionado = valor
                    else:
                        widget.disabled = not campo.get('editable', True)
                elif self.modo_actual == 'borrar':
                    widget.disabled = True
            
            self.id_seleccionado = id_valor
            mostrar_mensaje('info', 'Datos cargados para ID {}'.format(id_valor), self.output_mensajes)
            
        except Exception as e:
            mostrar_mensaje('error', 'Error cargando datos: {}'.format(str(e)), self.output_mensajes)
    
    def validar_formulario(self):
        """Valida todos los campos del formulario segun su configuracion"""
        errores = []
        valores = {}
        
        for campo in self.campos:
            nombre = campo['nombre']
            widget = self.widgets_campos[nombre]
            valor = widget.value
            
            requerido = campo.get('requerido', False)
            tipo = campo['tipo']
            min_val = campo.get('min')
            max_val = campo.get('max')
            es_id = campo.get('es_id', False)
            
            es_valido, mensaje = Validador.validar(
                valor, tipo, requerido, min_val, max_val,
                tabla=self.nombre_tabla if es_id else None,
                campo=nombre if es_id else None,
                id_actual=self.id_seleccionado if self.modo_actual == 'modificar' else None
            )
            
            if not es_valido:
                errores.append("<b>{}:</b> {}".format(campo.get('etiqueta', nombre), mensaje))
            else:
                valores[nombre] = valor
        
        return errores, valores
    
    def on_ejecutar(self, b):
        """Maneja la ejecucion segun el modo actual"""
        try:
            if self.modo_actual == 'insertar':
                self.ejecutar_insertar()
            elif self.modo_actual == 'modificar':
                self.ejecutar_modificar()
            elif self.modo_actual == 'borrar':
                self.ejecutar_borrar()
            elif self.modo_actual == 'consultar':
                self.ejecutar_consultar()
        except Exception as e:
            mostrar_mensaje('error', str(e), self.output_mensajes)
    
    def ejecutar_insertar(self):
        """Inserta nuevo registro con validaciones completas"""
        errores, valores = self.validar_formulario()
        
        if errores:
            mostrar_mensaje('error', "<br>".join(errores), self.output_mensajes)
            return
        
        # Construir SQL
        nombres = list(valores.keys())
        placeholders = ', '.join(['?' for _ in valores])
        sql = "INSERT INTO {} ({}) VALUES ({})".format(
            self.nombre_tabla, ', '.join(nombres), placeholders)
        
        try:
            cursor.execute(sql, list(valores.values()))
            conn.commit()
            mostrar_mensaje('exito', 'Registro insertado con ID: {}'.format(valores.get(nombres[0])), self.output_mensajes)
            self.limpiar_formulario()
            self.widgets_campos[nombres[0]].value = get_siguiente_id(self.nombre_tabla, nombres[0])
            self.mostrar_tabla_completa()
            # NOTIFICAR a otras tablas que hubo cambio
            if self.on_operacion_completada:
                self.on_operacion_completada(self.nombre_tabla, 'insertar')
        except sqlite3.IntegrityError as e:
            mostrar_mensaje('error', 'Error de integridad: {}'.format(str(e)), self.output_mensajes)
        except Exception as e:
            mostrar_mensaje('error', 'Error al insertar: {}'.format(str(e)), self.output_mensajes)
    
    def ejecutar_modificar(self):
        """Actualiza registro existente"""
        if not self.id_seleccionado:
            mostrar_mensaje('error', 'Primero cargue los datos con el boton [Cargar Datos]', self.output_mensajes)
            return
        
        errores, valores = self.validar_formulario()
        
        if errores:
            mostrar_mensaje('error', "<br>".join(errores), self.output_mensajes)
            return
        
        campo_id = next((c for c in self.campos if c.get('es_id', False)), None)
        if not campo_id:
            return
            
        id_nombre = campo_id['nombre']
        sets = []
        valores_update = []
        
        for nombre, valor in valores.items():
            if nombre != id_nombre:
                sets.append("{} = ?".format(nombre))
                valores_update.append(valor)
        
        valores_update.append(self.id_seleccionado)
        sql = "UPDATE {} SET {} WHERE {} = ?".format(
            self.nombre_tabla, ', '.join(sets), id_nombre)
        
        try:
            cursor.execute(sql, valores_update)
            conn.commit()
            mostrar_mensaje('exito', 'Registro ID {} actualizado correctamente'.format(self.id_seleccionado), self.output_mensajes)
            self.mostrar_tabla_completa()
            # NOTIFICAR a otras tablas que hubo cambio
            if self.on_operacion_completada:
                self.on_operacion_completada(self.nombre_tabla, 'modificar')
        except Exception as e:
            mostrar_mensaje('error', 'Error al actualizar: {}'.format(str(e)), self.output_mensajes)
    
    def ejecutar_borrar(self):
        """Soft delete con confirmacion"""
        if not self.id_seleccionado:
            mostrar_mensaje('error', 'Primero cargue los datos con el boton [Cargar Datos]', self.output_mensajes)
            return
        
        with self.output_confirmacion:
            clear_output()
            campo_id = next((c for c in self.campos if c.get('es_id', False)), None)
            
            display(widgets.HTML("""
                <div style='background:#fff3cd; padding:15px; border-radius:5px; margin:10px 0;'>
                    <h4 style='color:#856404; margin-top:0;'>Confirmar Borrado</h4>
                    <p>Esta seguro de que desea desactivar el registro con 
                    <b>ID {}</b> de la tabla <b>{}</b>?</p>
                    <p style='color:#856404; font-size:12px;'>Esta operacion realiza un borrado logico (soft delete). 
                    El registro permanecera en la base de datos pero marcado como inactivo.</p>
                </div>
            """.format(self.id_seleccionado, self.nombre_tabla)))
            
            btn_confirmar = widgets.Button(description='SI, desactivar', button_style='danger')
            btn_cancelar = widgets.Button(description='Cancelar', button_style='warning')
            
            def on_confirmar(b):
                try:
                    id_nombre = campo_id['nombre']
                    cursor.execute("UPDATE {} SET activo = 0 WHERE {} = ?".format(
                        self.nombre_tabla, id_nombre), (self.id_seleccionado,))
                    conn.commit()
                    mostrar_mensaje('exito', 'Registro ID {} desactivado correctamente'.format(self.id_seleccionado), self.output_mensajes)
                    self.id_seleccionado = None
                    self.limpiar_formulario()
                    with self.output_confirmacion:
                        clear_output()
                    self.mostrar_tabla_completa()
                    # NOTIFICAR a otras tablas que hubo cambio
                    if self.on_operacion_completada:
                        self.on_operacion_completada(self.nombre_tabla, 'borrar')
                except Exception as e:
                    mostrar_mensaje('error', 'Error al borrar: {}'.format(str(e)), self.output_mensajes)
            
            def on_cancelar(b):
                with self.output_confirmacion:
                    clear_output()
                mostrar_mensaje('info', 'Operacion cancelada', self.output_mensajes)
            
            btn_confirmar.on_click(on_confirmar)
            btn_cancelar.on_click(on_cancelar)
            
            display(widgets.HBox([btn_confirmar, btn_cancelar]))
    
    def ejecutar_consultar(self):
        """Consulta flexible con LIKE y soporte regex"""
        condiciones = []
        valores = []
        
        for campo in self.campos:
            nombre = campo['nombre']
            valor = self.widgets_campos[nombre].value
            tipo = campo['tipo']
            
            if not self.valor_significativo(campo, valor):
                continue
            
            valor_str = str(valor).strip()
            
            if tipo in ['texto', 'area']:
                if any(c in valor_str for c in ['*', '?', '.', '[', ']']):
                    like_pattern = valor_str.replace('*', '%').replace('?', '_').replace('.', '_')
                    condiciones.append("{} LIKE ?".format(nombre))
                    valores.append('%{}%'.format(like_pattern))
                else:
                    condiciones.append("{} LIKE ?".format(nombre))
                    valores.append('%{}%'.format(valor_str))
            else:
                condiciones.append("{} = ?".format(nombre))
                valores.append(valor)
        
        where = " AND ".join(condiciones) if condiciones else "1=1"
        sql = "SELECT * FROM {} WHERE {} AND activo = 1".format(self.nombre_tabla, where)
        
        try:
            df = pd.read_sql_query(sql, conn, params=valores)
            self.mostrar_dataframe(df, "Resultados: {} registros encontrados".format(len(df)))
            
            if len(df) == 0:
                mostrar_mensaje('advertencia', 'No se encontraron registros con esos criterios', self.output_mensajes)
            else:
                mostrar_mensaje('exito', 'Consulta exitosa: {} registros'.format(len(df)), self.output_mensajes)
        except Exception as e:
            mostrar_mensaje('error', 'Error en consulta: {}'.format(str(e)), self.output_mensajes)
    
    def valor_significativo(self, campo, valor):
        """Determina si un valor es significativo para busqueda"""
        if valor is None:
            return False
        if str(valor).strip() == '':
            return False
        if campo['tipo'] == 'fecha' and (valor == '' or str(valor).strip() == ''):
            return False
        if campo['tipo'] == 'checkbox':
            return False
        if campo['tipo'] in ['numero', 'decimal', 'slider', 'float_slider']:
            if valor == 0 or valor == 0.0:
                return False
        if campo['tipo'] == 'dropdown' and valor is None:
            return False
        if campo['tipo'] == 'radio' and valor is None:
            return False
        return True
    
    def on_limpiar(self, b):
        """Limpia formulario segun el modo actual"""
        if self.modo_actual == 'consultar':
            for campo in self.campos:
                widget = self.widgets_campos[campo['nombre']]
                self.limpiar_campo_consulta(campo, widget)
        else:
            self.limpiar_formulario()
        
        with self.output_confirmacion:
            clear_output()
        mostrar_mensaje('info', 'Formulario limpiado', self.output_mensajes)
    
    def on_exportar(self, b):
        """Exporta la tabla a CSV"""
        try:
            df = pd.read_sql_query("SELECT * FROM {} WHERE activo=1".format(self.nombre_tabla), conn)
            with self.output_mensajes:
                clear_output()
                display(widgets.HTML(crear_enlace_csv(df, "{}.csv".format(self.nombre_tabla))))
        except Exception as e:
            mostrar_mensaje('error', 'Error exportando: {}'.format(str(e)), self.output_mensajes)
    
    def mostrar_tabla_completa(self):
        """Muestra todos los registros activos de la tabla"""
        try:
            global conn
            if conn is None:
                mostrar_mensaje('error', 'Error: No hay conexion a la base de datos', self.output_mensajes)
                return
            
            sql = "SELECT * FROM {} WHERE activo = 1 LIMIT 100".format(self.nombre_tabla)
            df = pd.read_sql_query(sql, conn)
            titulo = "Tabla {} - {} registros activos".format(self.nombre_tabla, len(df))
            self.mostrar_dataframe(df, titulo)
            
        except Exception as e:
            error_msg = "Error mostrando tabla {}: {}".format(self.nombre_tabla, str(e))
            mostrar_mensaje('error', error_msg, self.output_mensajes)
            print(error_msg)
    
    def mostrar_dataframe(self, df, titulo):
        """Muestra un DataFrame en el output de tabla"""
        with self.output_tabla:
            clear_output()
            display(widgets.HTML("<h4>{}</h4>".format(titulo)))
            if len(df) > 0:
                display(df)
            else:
                display(widgets.HTML("<p style='color:#666;'>No hay registros activos en esta tabla</p>"))
    
    def limpiar_formulario(self):
        for campo in self.campos:
            self.resetear_valor(campo, self.widgets_campos[campo['nombre']])
        self.id_seleccionado = None
    
    def mostrar(self):
        return self.vista

print("Clase CRUDTabla (Version con Callback de Refresco) definida")

Clase CRUDTabla (Version con Callback de Refresco) definida


In [5]:
# ============================================
# CELDA 5: Clase CRUDMovimientos (Especializada con Refresco) - CORREGIDA
# ============================================

class CRUDMovimientos(CRUDTabla):
    def __init__(self, nombre_tabla, campos, on_operacion_completada=None):
        # Guardar callback antes de llamar al padre
        self._callback_externo = on_operacion_completada
        
        # Cargar opciones primero
        self._cargar_opciones()
        
        # Actualizar campos con opciones
        for campo in campos:
            if campo['nombre'] == 'id_producto':
                campo['opciones'] = self.opciones_productos
                campo['default'] = None
            elif campo['nombre'] == 'id_cliente':
                campo['opciones'] = self.opciones_clientes
                campo['default'] = None
            elif campo['nombre'] == 'id_proveedor':
                campo['opciones'] = self.opciones_proveedores
                campo['default'] = None
            elif campo['nombre'] == 'id_usuario':
                campo['opciones'] = self.opciones_usuarios
                campo['default'] = None
            elif campo['nombre'] == 'tipo':
                campo['default'] = None
        
        # Llamar al init de la clase padre
        super().__init__(nombre_tabla, campos, on_operacion_completada)
        
        # Configurar observadores para actualizacion automatica de stock
        self._configurar_observadores_stock()
    
    def _cargar_opciones(self):
        """Carga/Recarga todas las opciones de dropdowns desde la BD"""
        self.opciones_productos = get_productos()
        self.opciones_clientes = [("Ninguno", None)] + get_clientes()
        self.opciones_proveedores = [("Ninguno", None)] + get_proveedores()
        self.opciones_usuarios = get_usuarios()
    
    def refrescar_dropdowns(self):
        """Metodo publico para refrescar todos los dropdowns desde fuera"""
        self._cargar_opciones()
        
        # Actualizar widgets si ya existen
        if hasattr(self, 'widgets_campos'):
            if 'id_producto' in self.widgets_campos:
                self.widgets_campos['id_producto'].options = self.opciones_productos
            if 'id_cliente' in self.widgets_campos:
                self.widgets_campos['id_cliente'].options = self.opciones_clientes
            if 'id_proveedor' in self.widgets_campos:
                self.widgets_campos['id_proveedor'].options = self.opciones_proveedores
            if 'id_usuario' in self.widgets_campos:
                self.widgets_campos['id_usuario'].options = self.opciones_usuarios
    
    def _configurar_observadores_stock(self):
        """Configura observadores para actualizar stock automaticamente al cambiar cantidad o producto"""
        # Observar cambios en cantidad
        if 'cantidad' in self.widgets_campos:
            self.widgets_campos['cantidad'].observe(self._actualizar_stock_calculado, 'value')
        
        # Observar cambios en producto
        if 'id_producto' in self.widgets_campos:
            self.widgets_campos['id_producto'].observe(self._actualizar_stock_calculado, 'value')
        
        # Observar cambios en tipo (entrada/salida)
        if 'tipo' in self.widgets_campos:
            self.widgets_campos['tipo'].observe(self._actualizar_stock_calculado, 'value')
    
    def _actualizar_stock_calculado(self, change=None):
        """Calcula y muestra el stock anterior y posterior automaticamente"""
        try:
            # Obtener valores actuales
            id_producto = self.widgets_campos.get('id_producto', {}).value
            cantidad = self.widgets_campos.get('cantidad', {}).value
            tipo = self.widgets_campos.get('tipo', {}).value
            
            # Si no hay producto seleccionado, poner en 0
            if not id_producto:
                if 'stock_anterior' in self.widgets_campos:
                    self.widgets_campos['stock_anterior'].value = 0
                if 'stock_posterior' in self.widgets_campos:
                    self.widgets_campos['stock_posterior'].value = 0
                return
            
            # Obtener stock actual de la base de datos
            cursor.execute("SELECT stock_actual, precio_venta FROM productos WHERE id_producto = ?", (id_producto,))
            resultado = cursor.fetchone()
            
            if not resultado:
                return
            
            stock_actual, precio_venta = resultado
            
            # Calcular stock posterior
            if tipo == 'entrada':
                stock_posterior = stock_actual + (cantidad if cantidad else 0)
            else:
                stock_posterior = stock_actual - (cantidad if cantidad else 0)
            
            # Actualizar campos en pantalla
            if 'stock_anterior' in self.widgets_campos:
                self.widgets_campos['stock_anterior'].value = stock_actual
            
            if 'stock_posterior' in self.widgets_campos:
                self.widgets_campos['stock_posterior'].value = stock_posterior
            
            # Si es salida, actualizar precio unitario con precio de venta automaticamente
            if tipo == 'salida' and 'precio_unitario' in self.widgets_campos:
                self.widgets_campos['precio_unitario'].value = precio_venta
                
            # Calcular total
            if 'total' in self.widgets_campos:
                precio = self.widgets_campos['precio_unitario'].value if 'precio_unitario' in self.widgets_campos else 0
                cant = cantidad if cantidad else 0
                self.widgets_campos['total'].value = precio * cant
                
        except Exception as e:
            # Silenciar errores para no interrumpir la experiencia
            pass
    
    def configurar_campos_por_modo(self):
        """Sobrescribe para configurar observadores cuando cambia el modo y limpiar correctamente"""
        # Llamar al metodo padre primero
        super().configurar_campos_por_modo()
        
        # Refrescar dropdowns al cambiar de modo (importante para mantener sincronizacion)
        self.refrescar_dropdowns()
        
        # Si estamos en modo insertar, asegurar que los observadores esten activos
        if self.modo_actual == 'insertar':
            self._configurar_observadores_stock()
            # Forzar calculo inicial
            self._actualizar_stock_calculado()
    
    def limpiar_campo_consulta(self, campo, widget):
        """Sobrescribe para manejar especificamente los campos de movimientos - CORREGIDO"""
        tipo = campo['tipo']
        nombre = campo['nombre']
        
        try:
            if tipo in ['texto', 'area', 'fecha']:
                widget.value = ''
            elif tipo in ['numero', 'decimal']:
                # Para campos numericos en consulta, 0 significa "no filtrar"
                widget.value = 0
            elif tipo in ['slider', 'float_slider']:
                # Sliders: valor minimo definido o 0
                min_val = campo.get('min', 0)
                widget.value = min_val
            elif tipo == 'checkbox':
                widget.value = False
            elif tipo == 'dropdown':
                # Para dropdowns de movimientos, SIEMPRE poner None (sin seleccion)
                # Esto indica "no filtrar por este campo"
                try:
                    # Verificar si None es una opcion valida
                    opciones_vals = [opt[1] if isinstance(opt, tuple) else opt for opt in widget.options]
                    if None in opciones_vals:
                        widget.value = None
                    else:
                        # Si no hay None en opciones, usar primera opcion como "todos"
                        widget.value = widget.options[0][1] if isinstance(widget.options[0], tuple) else widget.options[0]
                except:
                    pass
            elif tipo == 'radio':
                # Para radio buttons en consulta, intentar poner None
                try:
                    widget.value = None
                except:
                    # Si no acepta None, dejar el valor actual (no se puede "limpiar" un radio)
                    pass
        except Exception as e:
            # Silenciar errores de limpieza
            pass
    
    def on_limpiar(self, b):
        """Sobrescribe limpiar para manejar especificamente el modo consulta en movimientos"""
        if self.modo_actual == 'consultar':
            # Limpiar todos los campos a valores "neutros" para consulta
            for campo in self.campos:
                widget = self.widgets_campos[campo['nombre']]
                self.limpiar_campo_consulta(campo, widget)
            
            # Limpiar tambien los outputs
            with self.output_confirmacion:
                clear_output()
            
            # Mostrar tabla completa nuevamente
            self.mostrar_tabla_completa()
            
            mostrar_mensaje('info', 'Formulario limpiado. Mostrando todos los registros.', self.output_mensajes)
        else:
            # Para otros modos, usar el comportamiento estandar
            self.limpiar_formulario()
            with self.output_confirmacion:
                clear_output()
            mostrar_mensaje('info', 'Formulario limpiado', self.output_mensajes)
    
    def ejecutar_insertar(self):
        """Sobrescribe insertar para manejar logica de inventario"""
        tipo = self.widgets_campos['tipo'].value
        id_producto = self.widgets_campos['id_producto'].value
        cantidad = self.widgets_campos['cantidad'].value
        precio_unitario = self.widgets_campos['precio_unitario'].value
        id_usuario = self.widgets_campos['id_usuario'].value
        
        # Validaciones basicas
        if not id_producto:
            mostrar_mensaje('error', 'Selecciona un producto', self.output_mensajes)
            return
        
        if cantidad <= 0:
            mostrar_mensaje('error', 'La cantidad debe ser mayor a 0', self.output_mensajes)
            return
        
        cursor.execute("SELECT stock_actual, precio_costo, precio_venta FROM productos WHERE id_producto = ?", 
                      (id_producto,))
        resultado = cursor.fetchone()
        if not resultado:
            mostrar_mensaje('error', 'Producto no encontrado', self.output_mensajes)
            return
            
        stock_actual, costo, venta = resultado
        
        if tipo == 'salida' and cantidad > stock_actual:
            mostrar_mensaje('error', 'Stock insuficiente. Disponible: {}'.format(stock_actual), self.output_mensajes)
            return
        
        # Calcular valores segun tipo
        if tipo == 'entrada':
            nuevo_stock = stock_actual + cantidad
            if stock_actual + cantidad > 0:
                nuevo_costo = ((stock_actual * costo) + (cantidad * precio_unitario)) / (stock_actual + cantidad)
            else:
                nuevo_costo = precio_unitario
            id_cliente = None
            id_proveedor = self.widgets_campos['id_proveedor'].value
            total = cantidad * precio_unitario
        else:
            nuevo_stock = stock_actual - cantidad
            nuevo_costo = costo
            precio_unitario = venta
            id_cliente = self.widgets_campos['id_cliente'].value
            id_proveedor = None
            total = cantidad * venta
        
        try:
            # Actualizar producto
            cursor.execute("UPDATE productos SET stock_actual = ?, precio_costo = ? WHERE id_producto = ?",
                          (nuevo_stock, nuevo_costo, id_producto))
            
            # Insertar movimiento (ID manual o autogenerado)
            id_movimiento = self.widgets_campos.get('id_movimiento', {}).value
            if id_movimiento and id_movimiento > 0:
                # Verificar si ID existe
                cursor.execute("SELECT id_movimiento FROM movimientos WHERE id_movimiento = ?", (id_movimiento,))
                if cursor.fetchone():
                    mostrar_mensaje('error', 'El ID {} ya existe en movimientos'.format(id_movimiento), self.output_mensajes)
                    conn.rollback()
                    return
            
            cursor.execute("""
                INSERT INTO movimientos 
                (id_movimiento, tipo, id_producto, id_cliente, id_proveedor, cantidad, precio_unitario, total, 
                 stock_anterior, stock_posterior, id_usuario, referencia_documento, motivo, activo)
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, 1)
            """, (id_movimiento if id_movimiento else None, tipo, id_producto, id_cliente, id_proveedor, 
                  cantidad if tipo == 'entrada' else -cantidad,
                  precio_unitario, total, stock_actual, nuevo_stock, id_usuario,
                  self.widgets_campos['referencia_documento'].value,
                  self.widgets_campos['motivo'].value))
            
            conn.commit()
            mostrar_mensaje('exito', 
                '{}: {} unidades. Stock: {} -> {}'.format(tipo.upper(), cantidad, stock_actual, nuevo_stock), 
                self.output_mensajes)
            self.limpiar_formulario()
            self.mostrar_tabla_completa()
            # NOTIFICAR a otras tablas (incluyendo que productos cambiaron de stock)
            if self.on_operacion_completada:
                self.on_operacion_completada(self.nombre_tabla, 'insertar')
        except Exception as e:
            conn.rollback()
            mostrar_mensaje('error', 'Error en transaccion: {}'.format(str(e)), self.output_mensajes)

print("Clase CRUDMovimientos definida (con sistema de refresco dinamico)")

Clase CRUDMovimientos definida (con sistema de refresco dinamico)


In [6]:
# ============================================
# CELDA 6: Configuracion de las 6 tablas con SISTEMA DE REFRESCO GLOBAL
# ============================================

# DICCIONARIO GLOBAL para almacenar referencias a los CRUDs y sus funciones de refresco
crud_registry = {}

def notificar_cambio_global(tabla_origen, tipo_operacion):
    """
    Funcion central que notifica a todas las tablas dependientes cuando hay cambios.
    Se llama automaticamente despues de insertar/modificar/borrar en cualquier tabla.
    """
    print("[SISTEMA] Cambio detectado en '{}' ({}). Actualizando dropdowns dependientes...".format(
        tabla_origen, tipo_operacion))
    
    # Si cambiaron CATEGORIAS -> actualizar Categorias (dropdown padre) y Productos
    if tabla_origen == 'categorias':
        if 'categorias' in crud_registry:
            crud_registry['categorias']['refrescar']()
            print("  -> Dropdowns de Categorias actualizados")
        if 'productos' in crud_registry:
            crud_registry['productos']['refrescar']()
            print("  -> Dropdowns de Productos actualizados")
    
    # Si cambiaron PROVEEDORES -> actualizar Productos y Movimientos
    if tabla_origen == 'proveedores':
        if 'productos' in crud_registry:
            crud_registry['productos']['refrescar']()
            print("  -> Dropdowns de Productos actualizados")
        if 'movimientos' in crud_registry:
            crud_registry['movimientos']['refrescar']()
            print("  -> Dropdowns de Movimientos actualizados")
    
    # Si cambiaron CLIENTES -> actualizar Movimientos
    if tabla_origen == 'clientes':
        if 'movimientos' in crud_registry:
            crud_registry['movimientos']['refrescar']()
            print("  -> Dropdowns de Movimientos actualizados")
    
    # Si cambiaron USUARIOS -> actualizar Movimientos
    if tabla_origen == 'usuarios':
        if 'movimientos' in crud_registry:
            crud_registry['movimientos']['refrescar']()
            print("  -> Dropdowns de Movimientos actualizados")
    
    # Si cambiaron PRODUCTOS -> actualizar Movimientos
    if tabla_origen == 'productos':
        if 'movimientos' in crud_registry:
            crud_registry['movimientos']['refrescar']()
            print("  -> Dropdowns de Movimientos actualizados")

# ============================================
# 1. CATEGORIAS - CORREGIDO: NIVEL EDITABLE Y DROPDOWN CON TODAS LAS CATEGORIAS
# ============================================
campos_categorias = [
    {'nombre': 'id_categoria', 'etiqueta': 'ID Categoria', 'tipo': 'numero', 'es_id': True, 
     'requerido': True, 'editable': True, 'ayuda': 'Sugerido: siguiente disponible. Debe ser unico.'},
    {'nombre': 'nombre', 'etiqueta': 'Nombre', 'tipo': 'texto', 'requerido': True, 
     'editable': True, 'ayuda': 'Nombre unico de la categoria'},
    {'nombre': 'descripcion', 'etiqueta': 'Descripcion', 'tipo': 'area', 
     'editable': True, 'ayuda': 'Descripcion detallada'},
    {'nombre': 'id_padre', 'etiqueta': 'Categoria Padre', 'tipo': 'dropdown', 
     'editable': True, 'opciones': [], 'default': None, 
     'ayuda': 'Selecciona la categoria padre (Ninguna = Raiz nivel 0)'},
    {'nombre': 'nivel', 'etiqueta': 'Nivel', 'tipo': 'slider', 'editable': True, 
     'min': 0, 'max': 5, 'default': 0, 
     'ayuda': 'Nivel en la jerarquia (0=Raiz, 1=Hijo, 2=Nieto, etc.)'},
    {'nombre': 'activo', 'etiqueta': 'Activo', 'tipo': 'checkbox', 'editable': True, 
     'default': True, 'ayuda': 'Categoria activa?'}
]

# Funcion para generar el arbol visual de categorias completo
def generar_arbol_categorias():
    try:
        cursor.execute("""
            SELECT c.id_categoria, c.nombre, c.nivel, c.id_padre, p.nombre as nombre_padre
            FROM categorias c
            LEFT JOIN categorias p ON c.id_padre = p.id_categoria
            WHERE c.activo = 1
            ORDER BY COALESCE(c.id_padre, c.id_categoria), c.nivel, c.nombre
        """)
        filas = cursor.fetchall()
        
        html = """
        <div style='background: #f8f9fa; padding: 10px; border-radius: 5px; 
                    border: 1px solid #dee2e6; margin: 5px 0; font-family: monospace; font-size: 12px;'>
            <h5 style='margin: 0 0 8px 0; color: #495057;'>📁 Estructura de Categorias Existentes</h5>
            <ul style='list-style: none; padding-left: 0; margin: 0;'>
        """
        
        for fila in filas:
            id_cat, nombre, nivel, id_padre, nombre_padre = fila
            
            indent = "  " * nivel  # Indentacion visual segun nivel
            icon = "📁" if nivel == 0 else "📂" if nivel == 1 else "📄"
            
            if nivel == 0:
                html += """
                <li style='margin: 4px 0; color: #0d6efd; font-weight: bold;'>
                    {}{} {} (ID:{})
                </li>
                """.format(indent, icon, nombre, id_cat)
            else:
                html += """
                <li style='margin: 2px 0; color: #6c757d;'>
                    {}{} {} (ID:{}) ← {}
                </li>
                """.format(indent, icon, nombre, id_cat, nombre_padre if nombre_padre else 'Raiz')
        
        html += """
            </ul>
            <p style='margin: 8px 0 0 0; font-size: 11px; color: #856404; background: #fff3cd; 
                      padding: 5px; border-radius: 3px;'>
                💡 Puedes crear categorias de cualquier nivel. Selecciona el padre correspondiente.
            </p>
        </div>
        """
        return html
    except Exception as e:
        return "<p style='color: red;'>Error cargando arbol: {}</p>".format(str(e))

# Funcion para refrescar dropdowns de categorias
def actualizar_dropdowns_categorias():
    """Actualiza el dropdown de categorias padre con TODAS las categorias disponibles"""
    try:
        opciones_padre = get_categorias_padre()
        crud_categorias.widgets_campos['id_padre'].options = opciones_padre
        print("  [Categorias] Dropdown actualizado: {} opciones disponibles".format(len(opciones_padre)))
    except Exception as e:
        print("  [Categorias] Error actualizando dropdown: {}".format(str(e)))

# Crear instancia de CRUD para categorias con callback de notificacion
crud_categorias = CRUDTabla('categorias', campos_categorias, on_operacion_completada=notificar_cambio_global)

# Actualizar opciones del dropdown de padre con TODAS las categorias
actualizar_dropdowns_categorias()

# Crear widget de referencia visual del arbol
widget_arbol_categorias = widgets.HTML(
    value=generar_arbol_categorias(),
    layout=widgets.Layout(width='100%', margin='10px 0')
)

# Insertar el widget de arbol despues del dropdown de id_padre en el formulario
campos_actuales = list(crud_categorias.area_formulario.children)
nuevo_area_formulario = []

for widget in campos_actuales:
    nuevo_area_formulario.append(widget)
    # Buscar el widget del dropdown de id_padre para insertar despues
    if widget == crud_categorias.widgets_campos['id_padre']:
        nuevo_area_formulario.append(widget_arbol_categorias)

crud_categorias.area_formulario.children = tuple(nuevo_area_formulario)

# Sobrescribir el metodo on_cambio_accion para refrescar dropdowns y arbol al cambiar de modo
_categorias_original_on_cambio = crud_categorias.on_cambio_accion

def categorias_on_cambio_con_refresh(change):
    # Llamar al metodo original
    _categorias_original_on_cambio(change)
    # Refrescar dropdowns
    actualizar_dropdowns_categorias()
    # Actualizar arbol visual
    widget_arbol_categorias.value = generar_arbol_categorias()

crud_categorias.on_cambio_accion = categorias_on_cambio_con_refresh

# Registrar en el sistema global con funcion de refresco
crud_registry['categorias'] = {
    'instancia': crud_categorias,
    'refrescar': actualizar_dropdowns_categorias
}

# ============================================
# 2. PROVEEDORES
# ============================================
campos_proveedores = [
    {'nombre': 'id_proveedor', 'etiqueta': 'ID Proveedor', 'tipo': 'numero', 'es_id': True,
     'requerido': True, 'editable': True, 'ayuda': 'Sugerido: siguiente disponible. Debe ser unico.'},
    {'nombre': 'razon_social', 'etiqueta': 'Razon Social', 'tipo': 'texto', 'requerido': True,
     'editable': True, 'ayuda': 'Nombre fiscal completo'},
    {'nombre': 'nombre_comercial', 'etiqueta': 'Nombre Comercial', 'tipo': 'texto',
     'editable': True, 'ayuda': 'Nombre comercial o de marca'},
    {'nombre': 'rfc', 'etiqueta': 'RFC', 'tipo': 'texto', 'requerido': True,
     'editable': True, 'ayuda': 'RFC mexicano (12-13 caracteres)'},
    {'nombre': 'direccion', 'etiqueta': 'Direccion', 'tipo': 'area',
     'editable': True, 'ayuda': 'Direccion completa'},
    {'nombre': 'telefono', 'etiqueta': 'Telefono', 'tipo': 'texto', 'requerido': True,
     'editable': True, 'ayuda': 'Numero de contacto'},
    {'nombre': 'email', 'etiqueta': 'Email', 'tipo': 'texto', 'requerido': True,
     'editable': True, 'ayuda': 'Correo electronico'},
    {'nombre': 'contacto_nombre', 'etiqueta': 'Nombre Contacto', 'tipo': 'texto',
     'editable': True, 'ayuda': 'Persona de contacto principal'},
    {'nombre': 'dias_credito', 'etiqueta': 'Dias Credito', 'tipo': 'slider', 'editable': True,
     'min': 0, 'max': 90, 'default': 0, 'ayuda': 'Dias de credito permitidos'},
    {'nombre': 'activo', 'etiqueta': 'Activo', 'tipo': 'checkbox', 'editable': True,
     'default': True, 'ayuda': 'Proveedor activo?'},
    {'nombre': 'fecha_registro', 'etiqueta': 'Fecha Registro', 'tipo': 'fecha',
     'editable': False, 'ayuda': 'Fecha automatica de registro'}
]

crud_proveedores = CRUDTabla('proveedores', campos_proveedores, on_operacion_completada=notificar_cambio_global)

crud_registry['proveedores'] = {
    'instancia': crud_proveedores,
    'refrescar': lambda: None
}

# ============================================
# 3. CLIENTES
# ============================================
campos_clientes = [
    {'nombre': 'id_cliente', 'etiqueta': 'ID Cliente', 'tipo': 'numero', 'es_id': True,
     'requerido': True, 'editable': True, 'ayuda': 'Sugerido: siguiente disponible. Debe ser unico.'},
    {'nombre': 'tipo_cliente', 'etiqueta': 'Tipo', 'tipo': 'radio', 'requerido': True,
     'editable': True, 'opciones': [('Persona', 'persona'), ('Empresa', 'empresa')], 
     'default': 'persona', 'ayuda': 'Tipo de cliente'},
    {'nombre': 'nombre', 'etiqueta': 'Nombre', 'tipo': 'texto', 'requerido': True,
     'editable': True, 'ayuda': 'Nombre o razon social'},
    {'nombre': 'apellidos', 'etiqueta': 'Apellidos', 'tipo': 'texto',
     'editable': True, 'ayuda': 'Apellidos (solo para personas)'},
    {'nombre': 'rfc', 'etiqueta': 'RFC', 'tipo': 'texto', 'requerido': True,
     'editable': True, 'ayuda': 'RFC mexicano'},
    {'nombre': 'email', 'etiqueta': 'Email', 'tipo': 'texto', 'requerido': True,
     'editable': True, 'ayuda': 'Correo electronico'},
    {'nombre': 'telefono', 'etiqueta': 'Telefono', 'tipo': 'texto', 'requerido': True,
     'editable': True, 'ayuda': 'Numero de contacto'},
    {'nombre': 'direccion', 'etiqueta': 'Direccion', 'tipo': 'area',
     'editable': True, 'ayuda': 'Direccion fiscal'},
    {'nombre': 'direccion_envio', 'etiqueta': 'Direccion Envio', 'tipo': 'area',
     'editable': True, 'ayuda': 'Direccion de entrega (si es diferente)'},
    {'nombre': 'limite_credito', 'etiqueta': 'Limite Credito', 'tipo': 'slider', 'editable': True,
     'min': 0, 'max': 100000, 'step': 1000, 'default': 0, 'ayuda': 'Limite de credito en pesos'},
    {'nombre': 'saldo_pendiente', 'etiqueta': 'Saldo', 'tipo': 'decimal', 'editable': False,
     'default': 0, 'ayuda': 'Saldo pendiente (calculado)'},
    {'nombre': 'activo', 'etiqueta': 'Activo', 'tipo': 'checkbox', 'editable': True,
     'default': True, 'ayuda': 'Cliente activo?'},
    {'nombre': 'fecha_registro', 'etiqueta': 'Fecha Registro', 'tipo': 'fecha',
     'editable': False, 'ayuda': 'Fecha automatica de registro'}
]

crud_clientes = CRUDTabla('clientes', campos_clientes, on_operacion_completada=notificar_cambio_global)

crud_registry['clientes'] = {
    'instancia': crud_clientes,
    'refrescar': lambda: None
}

# ============================================
# 4. USUARIOS
# ============================================
campos_usuarios = [
    {'nombre': 'id_usuario', 'etiqueta': 'ID Usuario', 'tipo': 'numero', 'es_id': True,
     'requerido': True, 'editable': True, 'ayuda': 'Sugerido: siguiente disponible. Debe ser unico.'},
    {'nombre': 'username', 'etiqueta': 'Username', 'tipo': 'texto', 'requerido': True,
     'editable': True, 'ayuda': 'Nombre de usuario unico (sin espacios)'},
    {'nombre': 'password_hash', 'etiqueta': 'Password', 'tipo': 'texto', 'requerido': True,
     'editable': True, 'default': 'password123', 'ayuda': 'Contraseña (texto plano - DEMO: en produccion usar hash)'},
    {'nombre': 'nombre_completo', 'etiqueta': 'Nombre Completo', 'tipo': 'texto', 'requerido': True,
     'editable': True, 'ayuda': 'Nombre completo del usuario'},
    {'nombre': 'email', 'etiqueta': 'Email', 'tipo': 'texto', 'requerido': True,
     'editable': True, 'ayuda': 'Correo institucional'},
    {'nombre': 'rol', 'etiqueta': 'Rol', 'tipo': 'dropdown', 'requerido': True, 'editable': True,
     'opciones': [('Admin', 'admin'), ('Gerente', 'gerente'), ('Vendedor', 'vendedor'), 
                  ('Almacenista', 'almacenista'), ('Auditor', 'auditor')], 
     'default': 'vendedor', 'ayuda': 'Rol dentro del sistema'},
    {'nombre': 'activo', 'etiqueta': 'Activo', 'tipo': 'checkbox', 'editable': True,
     'default': True, 'ayuda': 'Usuario activo?'},
    {'nombre': 'ultimo_acceso', 'etiqueta': 'Ultimo Acceso', 'tipo': 'fecha',
     'editable': False, 'ayuda': 'Ultima fecha de acceso (automatico)'},
    {'nombre': 'intentos_fallidos', 'etiqueta': 'Intentos Fallidos', 'tipo': 'numero', 'editable': False,
     'default': 0, 'ayuda': 'Contador de intentos fallidos (automatico)'},
    {'nombre': 'fecha_registro', 'etiqueta': 'Fecha Registro', 'tipo': 'fecha',
     'editable': False, 'ayuda': 'Fecha de creacion (automatica)'}
]

crud_usuarios = CRUDTabla('usuarios', campos_usuarios, on_operacion_completada=notificar_cambio_global)

crud_registry['usuarios'] = {
    'instancia': crud_usuarios,
    'refrescar': lambda: None
}

# ============================================
# 5. PRODUCTOS - CORREGIDO: codigo_barras opcional, guarda None si vacio
# ============================================
def actualizar_dropdowns_productos():
    """Actualiza las opciones de categorias y proveedores en tiempo real"""
    try:
        # Actualizar categorias (TODAS las categorias activas)
        opciones_cats = get_categorias()
        crud_productos.widgets_campos['id_categoria'].options = opciones_cats
        
        # Actualizar proveedores
        opciones_provs = get_proveedores()
        crud_productos.widgets_campos['id_proveedor'].options = opciones_provs
        
        print("  [Productos] Dropdowns actualizados: {} categorias, {} proveedores".format(
            len(opciones_cats), len(opciones_provs)))
    except Exception as e:
        print("  [Productos] Error actualizando dropdowns: {}".format(str(e)))

campos_productos = [
    {'nombre': 'id_producto', 'etiqueta': 'ID Producto', 'tipo': 'numero', 'es_id': True,
     'requerido': True, 'editable': True, 'ayuda': 'Sugerido: siguiente disponible. Debe ser unico.'},
    {'nombre': 'sku', 'etiqueta': 'SKU', 'tipo': 'texto', 'requerido': True,
     'editable': True, 'ayuda': 'Codigo SKU unico (ej: LAP-001)'},
    {'nombre': 'codigo_barras', 'etiqueta': 'Codigo Barras', 'tipo': 'texto',
     'editable': True, 'default': '', 'ayuda': 'Codigo de barras EAN/UPC (opcional)'},
    {'nombre': 'nombre', 'etiqueta': 'Nombre', 'tipo': 'texto', 'requerido': True,
     'editable': True, 'ayuda': 'Nombre descriptivo del producto'},
    {'nombre': 'descripcion', 'etiqueta': 'Descripcion', 'tipo': 'area',
     'editable': True, 'ayuda': 'Descripcion detallada'},
    {'nombre': 'id_categoria', 'etiqueta': 'Categoria', 'tipo': 'dropdown', 'requerido': True,
     'editable': True, 'opciones': [], 'default': None, 'ayuda': 'Categoria del producto (cualquier nivel)'},
    {'nombre': 'id_proveedor', 'etiqueta': 'Proveedor', 'tipo': 'dropdown', 'requerido': True,
     'editable': True, 'opciones': [], 'default': None, 'ayuda': 'Proveedor principal'},
    {'nombre': 'precio_costo', 'etiqueta': 'Precio Costo', 'tipo': 'float_slider', 'requerido': True,
     'editable': True, 'min': 0, 'max': 50000, 'step': 0.01, 'default': 0, 'ayuda': 'Precio de compra'},
    {'nombre': 'precio_venta', 'etiqueta': 'Precio Venta', 'tipo': 'float_slider', 'requerido': True,
     'editable': True, 'min': 0, 'max': 50000, 'step': 0.01, 'default': 0, 'ayuda': 'Precio de venta'},
    {'nombre': 'stock_minimo', 'etiqueta': 'Stock Minimo', 'tipo': 'numero', 'requerido': True,
     'editable': True, 'default': 0, 'ayuda': 'Nivel minimo de inventario'},
    {'nombre': 'stock_maximo', 'etiqueta': 'Stock Maximo', 'tipo': 'numero', 'requerido': True,
     'editable': True, 'default': 0, 'ayuda': 'Capacidad maxima de almacen'},
    {'nombre': 'stock_actual', 'etiqueta': 'Stock Actual', 'tipo': 'numero', 'editable': False,
     'default': 0, 'ayuda': 'Inventario actual (automatico por movimientos)'},
    {'nombre': 'unidad_medida', 'etiqueta': 'Unidad', 'tipo': 'dropdown', 'editable': True,
     'opciones': [('Pieza', 'pieza'), ('Kilogramo', 'kg'), ('Litro', 'litro'), ('Metro', 'metro')], 
     'default': 'pieza', 'ayuda': 'Unidad de medida'},
    {'nombre': 'ubicacion_almacen', 'etiqueta': 'Ubicacion', 'tipo': 'texto',
     'editable': True, 'ayuda': 'Ubicacion fisica (ej: A-01-05)'},
    {'nombre': 'activo', 'etiqueta': 'Activo', 'tipo': 'checkbox', 'editable': True,
     'default': True, 'ayuda': 'Producto activo?'},
    {'nombre': 'fecha_registro', 'etiqueta': 'Fecha Registro', 'tipo': 'fecha',
     'editable': False, 'ayuda': 'Fecha de alta (automatica)'}
]

# Sobrescribir CRUDTabla para manejar codigo_barras vacio como None
class CRUDProductos(CRUDTabla):
    def ejecutar_insertar(self):
        """Inserta nuevo registro convirtiendo codigo_barras vacio a None"""
        errores, valores = self.validar_formulario()
        
        if errores:
            mostrar_mensaje('error', "<br>".join(errores), self.output_mensajes)
            return
        
        # Convertir codigo_barras vacio a None
        if 'codigo_barras' in valores:
            if valores['codigo_barras'] is None or str(valores['codigo_barras']).strip() == '':
                valores['codigo_barras'] = None
        
        # Construir SQL
        nombres = list(valores.keys())
        placeholders = ', '.join(['?' for _ in valores])
        sql = "INSERT INTO {} ({}) VALUES ({})".format(
            self.nombre_tabla, ', '.join(nombres), placeholders)
        
        try:
            cursor.execute(sql, list(valores.values()))
            conn.commit()
            mostrar_mensaje('exito', 'Registro insertado con ID: {}'.format(valores.get(nombres[0])), self.output_mensajes)
            self.limpiar_formulario()
            self.widgets_campos[nombres[0]].value = get_siguiente_id(self.nombre_tabla, nombres[0])
            self.mostrar_tabla_completa()
            if self.on_operacion_completada:
                self.on_operacion_completada(self.nombre_tabla, 'insertar')
        except sqlite3.IntegrityError as e:
            mostrar_mensaje('error', 'Error de integridad: {}'.format(str(e)), self.output_mensajes)
        except Exception as e:
            mostrar_mensaje('error', 'Error al insertar: {}'.format(str(e)), self.output_mensajes)
    
    def ejecutar_modificar(self):
        """Actualiza registro existente convirtiendo codigo_barras vacio a None"""
        if not self.id_seleccionado:
            mostrar_mensaje('error', 'Primero cargue los datos con el boton [Cargar Datos]', self.output_mensajes)
            return
        
        errores, valores = self.validar_formulario()
        
        if errores:
            mostrar_mensaje('error', "<br>".join(errores), self.output_mensajes)
            return
        
        # Convertir codigo_barras vacio a None
        if 'codigo_barras' in valores:
            if valores['codigo_barras'] is None or str(valores['codigo_barras']).strip() == '':
                valores['codigo_barras'] = None
        
        campo_id = next((c for c in self.campos if c.get('es_id', False)), None)
        if not campo_id:
            return
            
        id_nombre = campo_id['nombre']
        sets = []
        valores_update = []
        
        for nombre, valor in valores.items():
            if nombre != id_nombre:
                sets.append("{} = ?".format(nombre))
                valores_update.append(valor)
        
        valores_update.append(self.id_seleccionado)
        sql = "UPDATE {} SET {} WHERE {} = ?".format(
            self.nombre_tabla, ', '.join(sets), id_nombre)
        
        try:
            cursor.execute(sql, valores_update)
            conn.commit()
            mostrar_mensaje('exito', 'Registro ID {} actualizado correctamente'.format(self.id_seleccionado), self.output_mensajes)
            self.mostrar_tabla_completa()
            if self.on_operacion_completada:
                self.on_operacion_completada(self.nombre_tabla, 'modificar')
        except Exception as e:
            mostrar_mensaje('error', 'Error al actualizar: {}'.format(str(e)), self.output_mensajes)

crud_productos = CRUDProductos('productos', campos_productos, on_operacion_completada=notificar_cambio_global)

# Actualizar dropdowns al inicio
actualizar_dropdowns_productos()

# Sobrescribir el metodo on_cambio_accion para refrescar dropdowns al cambiar de modo
_productos_original_on_cambio = crud_productos.on_cambio_accion

def productos_on_cambio_con_refresh(change):
    _productos_original_on_cambio(change)
    actualizar_dropdowns_productos()

crud_productos.on_cambio_accion = productos_on_cambio_con_refresh

# Registrar en el sistema global con funcion de refresco
crud_registry['productos'] = {
    'instancia': crud_productos,
    'refrescar': actualizar_dropdowns_productos
}

# ============================================
# 6. MOVIMIENTOS
# ============================================
def actualizar_dropdowns_movimientos():
    """Actualiza las opciones de productos, clientes, proveedores y usuarios en tiempo real"""
    try:
        # Actualizar productos
        opciones_prods = get_productos()
        crud_movimientos.widgets_campos['id_producto'].options = opciones_prods
        
        # Actualizar clientes (con opcion "Ninguno")
        opciones_clis = [("Ninguno", None)] + get_clientes()
        crud_movimientos.widgets_campos['id_cliente'].options = opciones_clis
        
        # Actualizar proveedores (con opcion "Ninguno")
        opciones_provs = [("Ninguno", None)] + get_proveedores()
        crud_movimientos.widgets_campos['id_proveedor'].options = opciones_provs
        
        # Actualizar usuarios
        opciones_usrs = get_usuarios()
        crud_movimientos.widgets_campos['id_usuario'].options = opciones_usrs
        
        print("  [Movimientos] Dropdowns actualizados: {} productos, {} clientes, {} proveedores, {} usuarios".format(
            len(opciones_prods), len(opciones_clis)-1, len(opciones_provs)-1, len(opciones_usrs)))
    except Exception as e:
        print("  [Movimientos] Error actualizando dropdowns: {}".format(str(e)))

campos_movimientos = [
    {'nombre': 'id_movimiento', 'etiqueta': 'ID Movimiento', 'tipo': 'numero', 'es_id': True,
     'requerido': False, 'editable': True, 'ayuda': 'Opcional. Si se deja en 0, se asigna automatico.'},
    {'nombre': 'tipo', 'etiqueta': 'Tipo', 'tipo': 'radio', 'requerido': True, 'editable': True,
     'opciones': [('Entrada', 'entrada'), ('Salida', 'salida')], 'default': 'entrada', 
     'ayuda': 'Tipo de movimiento de inventario'},
    {'nombre': 'id_producto', 'etiqueta': 'Producto', 'tipo': 'dropdown', 'requerido': True,
     'editable': True, 'opciones': [], 'default': None, 'ayuda': 'Producto a mover'},
    {'nombre': 'id_cliente', 'etiqueta': 'Cliente (salidas)', 'tipo': 'dropdown', 'editable': True,
     'opciones': [], 'default': None, 'ayuda': 'Cliente destino (solo para salidas)'},
    {'nombre': 'id_proveedor', 'etiqueta': 'Proveedor (entradas)', 'tipo': 'dropdown', 'editable': True,
     'opciones': [], 'default': None, 'ayuda': 'Proveedor origen (solo para entradas)'},
    {'nombre': 'cantidad', 'etiqueta': 'Cantidad', 'tipo': 'slider', 'requerido': True, 'editable': True,
     'min': 1, 'max': 100, 'default': 1, 'ayuda': 'Cantidad a mover'},
    {'nombre': 'precio_unitario', 'etiqueta': 'Precio Unitario', 'tipo': 'float_slider', 'editable': True,
     'min': 0, 'max': 50000, 'step': 0.01, 'default': 0, 'ayuda': 'Precio de la transaccion'},
    {'nombre': 'total', 'etiqueta': 'Total Calculado', 'tipo': 'decimal', 'editable': False,
     'default': 0, 'ayuda': 'Total automatico (cantidad x precio)'},
    {'nombre': 'stock_anterior', 'etiqueta': 'Stock Ant.', 'tipo': 'numero', 'editable': False,
     'default': 0, 'ayuda': 'Inventario antes del movimiento'},
    {'nombre': 'stock_posterior', 'etiqueta': 'Stock Post.', 'tipo': 'numero', 'editable': False,
     'default': 0, 'ayuda': 'Inventario despues del movimiento'},
    {'nombre': 'referencia_documento', 'etiqueta': 'Referencia', 'tipo': 'texto', 'editable': True,
     'ayuda': 'Numero de factura, remision, etc.'},
    {'nombre': 'motivo', 'etiqueta': 'Motivo', 'tipo': 'area', 'editable': True,
     'ayuda': 'Justificacion del movimiento'},
    {'nombre': 'id_usuario', 'etiqueta': 'Responsable', 'tipo': 'dropdown', 'requerido': True,
     'editable': True, 'opciones': [], 'default': None, 'ayuda': 'Usuario que realiza el movimiento'},
    {'nombre': 'fecha_movimiento', 'etiqueta': 'Fecha/Hora', 'tipo': 'fecha', 'editable': False,
     'ayuda': 'Fecha y hora automatica'}
]

crud_movimientos = CRUDMovimientos('movimientos', campos_movimientos, on_operacion_completada=notificar_cambio_global)

# Actualizar dropdowns al inicio
actualizar_dropdowns_movimientos()

# Sobrescribir el metodo on_cambio_accion para refrescar dropdowns al cambiar de modo
_movimientos_original_on_cambio = crud_movimientos.on_cambio_accion

def movimientos_on_cambio_con_refresh(change):
    _movimientos_original_on_cambio(change)
    actualizar_dropdowns_movimientos()

crud_movimientos.on_cambio_accion = movimientos_on_cambio_con_refresh

# Registrar en el sistema global con funcion de refresco
crud_registry['movimientos'] = {
    'instancia': crud_movimientos,
    'refrescar': actualizar_dropdowns_movimientos
}

print("=" * 70)
print("SISTEMA CONFIGURADO - DROPDOWNS CON TODAS LAS CATEGORIAS DISPONIBLES")
print("=" * 70)
print("Cambios principales:")
print("  • get_categorias_padre() ahora muestra TODAS las categorias (no solo nivel 0)")
print("  • Jerarquia visual en el dropdown: Padre → Hijo (ID) [Nivel]")
print("  • Nivel editable (0-5) para permitir cualquier profundidad de jerarquia")
print("  • Arbol visual actualizado dinamicamente")
print("  • codigo_barras ahora es OPCIONAL (sin UNIQUE, sin requerido)")
print("=" * 70)


  [Categorias] Dropdown actualizado: 55 opciones disponibles
  [Productos] Dropdowns actualizados: 54 categorias, 3 proveedores
  [Movimientos] Dropdowns actualizados: 5 productos, 3 clientes, 3 proveedores, 3 usuarios
SISTEMA CONFIGURADO - DROPDOWNS CON TODAS LAS CATEGORIAS DISPONIBLES
Cambios principales:
  • get_categorias_padre() ahora muestra TODAS las categorias (no solo nivel 0)
  • Jerarquia visual en el dropdown: Padre → Hijo (ID) [Nivel]
  • Nivel editable (0-5) para permitir cualquier profundidad de jerarquia
  • Arbol visual actualizado dinamicamente
  • codigo_barras ahora es OPCIONAL (sin UNIQUE, sin requerido)


In [7]:
# ============================================
# CELDA 7: Consola SQL Libre (con Refresco Automatico) - CORREGIDA
# ============================================

class ConsolaSQL:
    def __init__(self):
        self.output_resultado = widgets.Output()
        self.historial = []
        self.indice_historial = -1
        
        self.crear_interfaz()
    
    def crear_interfaz(self):
        # Area de texto para SQL
        self.textarea_sql = widgets.Textarea(
            value='',
            placeholder='-- Escribe tu consulta SQL aqui\n-- Ejemplos:\n-- SELECT * FROM productos WHERE precio_venta > 5000\n-- SELECT nombre, stock_actual FROM productos WHERE stock_actual < stock_minimo\n-- SELECT c.nombre, COUNT(p.id_producto) FROM categorias c LEFT JOIN productos p ON c.id_categoria = p.id_categoria GROUP BY c.id_categoria',
            layout=widgets.Layout(width='100%', height='150px'),
            style={'description_width': 'initial'}
        )
        
        # Botones con Unicode explicito
        self.btn_ejecutar = widgets.Button(
            description='\u25B6 Ejecutar SQL',
            button_style='primary',
            layout=widgets.Layout(width='150px')
        )
        self.btn_ejecutar.on_click(self.ejecutar_sql)
        
        self.btn_limpiar = widgets.Button(
            description='\u2717 Limpiar',
            button_style='warning',
            layout=widgets.Layout(width='120px')
        )
        self.btn_limpiar.on_click(self.limpiar_sql)
        
        self.btn_reset_datos = widgets.Button(
            description='\u21BB Resetear Datos',
            button_style='danger',
            layout=widgets.Layout(width='150px'),
            tooltip='Restaura los datos de prueba originales'
        )
        self.btn_reset_datos.on_click(self.resetear_datos)
        
        self.btn_anterior = widgets.Button(
            description='\u2191 Anterior',
            layout=widgets.Layout(width='100px')
        )
        self.btn_anterior.on_click(self.historial_anterior)
        
        self.btn_siguiente = widgets.Button(
            description='\u2193 Siguiente',
            layout=widgets.Layout(width='100px')
        )
        self.btn_siguiente.on_click(self.historial_siguiente)
        
        # Dropdown de tablas para ayuda
        self.dd_tablas = widgets.Dropdown(
            options=[''] + ['categorias', 'proveedores', 'clientes', 'usuarios', 'productos', 'movimientos'],
            description='Tablas:',
            layout=widgets.Layout(width='200px')
        )
        self.dd_tablas.observe(self.on_seleccion_tabla, 'value')
        
        # Layout de controles
        controles = widgets.HBox([
            self.btn_ejecutar,
            self.btn_limpiar,
            self.btn_reset_datos,
            widgets.HTML("<span style='margin: 0 10px;'>|</span>"),
            self.btn_anterior,
            self.btn_siguiente,
            widgets.HTML("<span style='margin: 0 10px;'>|</span>"),
            self.dd_tablas
        ], layout=widgets.Layout(margin='10px 0'))
        
        # Panel de ayuda rapida
        self.ayuda = widgets.HTML("""
            <div style='background: #e9ecef; padding: 10px; border-radius: 5px; font-size: 12px; color: #495057;'>
                <b>Tip:</b> SELECT, INSERT, UPDATE, DELETE permitidos | 
                Usa <code>activo = 1</code> para ver registros activos | 
                Tablas: categorias, proveedores, clientes, usuarios, productos, movimientos
            </div>
        """)
        
        self.vista = widgets.VBox([
            widgets.HTML("<h3 style='color: #495057; margin-top: 0;'>Consola SQL Libre</h3>"),
            self.ayuda,
            self.textarea_sql,
            controles,
            self.output_resultado
        ], layout=widgets.Layout(padding='15px', border='2px solid #dee2e6', 
                                border_radius='8px', margin='20px 0 0 0'))
    
    def on_seleccion_tabla(self, change):
        if change['new']:
            sql_actual = self.textarea_sql.value
            if sql_actual and not sql_actual.endswith(' '):
                sql_actual += ' '
            self.textarea_sql.value = sql_actual + change['new']
            self.dd_tablas.value = ''
    
    def ejecutar_sql(self, b):
        sql = self.textarea_sql.value.strip()
        
        if not sql:
            self.mostrar_error("Escribe una consulta SQL")
            return
        
        # Guardar en historial
        if not self.historial or self.historial[-1] != sql:
            self.historial.append(sql)
            self.indice_historial = len(self.historial) - 1
        
        sql_upper = sql.upper().strip()
        
        try:
            with self.output_resultado:
                clear_output()
                
                if sql_upper.startswith('SELECT'):
                    df = pd.read_sql_query(sql, conn)
                    print("Consulta exitosa: {} filas".format(len(df)))
                    print("-" * 50)
                    display(df)
                    
                elif sql_upper.startswith(('INSERT', 'UPDATE', 'DELETE')):
                    cursor.execute(sql)
                    conn.commit()
                    filas_afectadas = cursor.rowcount
                    print("Operacion exitosa")
                    print("Filas afectadas: {}".format(filas_afectadas))
                    
                    if sql_upper.startswith('INSERT'):
                        print("Ultimo ID generado: {}".format(cursor.lastrowid))
                    
                    # DETECTAR que tabla fue modificada y notificar
                    tabla_modificada = self._detectar_tabla(sql_upper)
                    if tabla_modificada and tabla_modificada in crud_registry:
                        print("\n[SISTEMA] Detectado cambio en '{}'. Sincronizando dropdowns...".format(tabla_modificada))
                        notificar_cambio_global(tabla_modificada, 'sql_directo')
                    
                else:
                    cursor.execute(sql)
                    conn.commit()
                    print("Comando ejecutado exitosamente")
                    
        except Exception as e:
            self.mostrar_error("Error SQL: {}".format(str(e)))
    
    def _detectar_tabla(self, sql_upper):
        """Detecta la tabla afectada en una consulta SQL"""
        tablas = ['categorias', 'proveedores', 'clientes', 'usuarios', 'productos', 'movimientos']
        for tabla in tablas:
            if tabla.upper() in sql_upper:
                return tabla
        return None
    
    def mostrar_error(self, mensaje):
        with self.output_resultado:
            clear_output()
            display(widgets.HTML(
                "<div style='color: #721c24; background: #f8d7da; padding: 10px; "
                "border-radius: 5px;'><b>Error:</b> {}</div>".format(mensaje)
            ))
    
    def limpiar_sql(self, b):
        self.textarea_sql.value = ''
        with self.output_resultado:
            clear_output()
    
    def historial_anterior(self, b):
        if self.historial and self.indice_historial > 0:
            self.indice_historial -= 1
            self.textarea_sql.value = self.historial[self.indice_historial]
    
    def historial_siguiente(self, b):
        if self.historial and self.indice_historial < len(self.historial) - 1:
            self.indice_historial += 1
            self.textarea_sql.value = self.historial[self.indice_historial]
    
    def resetear_datos(self, b):
        """Restaura datos de prueba originales"""
        global conn, cursor
        
        try:
            conn.close()
            conn = sqlite3.connect(':memory:')
            cursor = conn.cursor()
            
            # Reejecutar script de creacion (CON activo en movimientos)
            cursor.executescript("""
                CREATE TABLE categorias (
                    id_categoria INTEGER PRIMARY KEY,
                    nombre TEXT NOT NULL UNIQUE,
                    descripcion TEXT,
                    id_padre INTEGER,
                    nivel INTEGER DEFAULT 0,
                    activo INTEGER DEFAULT 1,
                    FOREIGN KEY (id_padre) REFERENCES categorias(id_categoria)
                );
                CREATE TABLE proveedores (
                    id_proveedor INTEGER PRIMARY KEY,
                    razon_social TEXT NOT NULL,
                    nombre_comercial TEXT,
                    rfc TEXT UNIQUE,
                    direccion TEXT,
                    telefono TEXT,
                    email TEXT,
                    contacto_nombre TEXT,
                    dias_credito INTEGER DEFAULT 0,
                    activo INTEGER DEFAULT 1,
                    fecha_registro TEXT DEFAULT CURRENT_TIMESTAMP
                );
                CREATE TABLE clientes (
                    id_cliente INTEGER PRIMARY KEY,
                    tipo_cliente TEXT CHECK(tipo_cliente IN ('persona', 'empresa')),
                    nombre TEXT NOT NULL,
                    apellidos TEXT,
                    rfc TEXT UNIQUE,
                    email TEXT,
                    telefono TEXT,
                    direccion TEXT,
                    direccion_envio TEXT,
                    limite_credito REAL DEFAULT 0,
                    saldo_pendiente REAL DEFAULT 0,
                    activo INTEGER DEFAULT 1,
                    fecha_registro TEXT DEFAULT CURRENT_TIMESTAMP
                );
                CREATE TABLE usuarios (
                    id_usuario INTEGER PRIMARY KEY,
                    username TEXT NOT NULL UNIQUE,
                    password_hash TEXT NOT NULL,
                    nombre_completo TEXT NOT NULL,
                    email TEXT NOT NULL UNIQUE,
                    rol TEXT CHECK(rol IN ('admin', 'gerente', 'vendedor', 'almacenista', 'auditor')),
                    activo INTEGER DEFAULT 1,
                    ultimo_acceso TEXT,
                    intentos_fallidos INTEGER DEFAULT 0,
                    fecha_registro TEXT DEFAULT CURRENT_TIMESTAMP
                );
                CREATE TABLE productos (
                    id_producto INTEGER PRIMARY KEY,
                    sku TEXT NOT NULL UNIQUE,
                    codigo_barras TEXT UNIQUE,
                    nombre TEXT NOT NULL,
                    descripcion TEXT,
                    id_categoria INTEGER,
                    id_proveedor INTEGER,
                    precio_costo REAL DEFAULT 0,
                    precio_venta REAL DEFAULT 0,
                    stock_minimo INTEGER DEFAULT 0,
                    stock_maximo INTEGER DEFAULT 0,
                    stock_actual INTEGER DEFAULT 0,
                    unidad_medida TEXT DEFAULT 'pieza',
                    ubicacion_almacen TEXT,
                    activo INTEGER DEFAULT 1,
                    fecha_registro TEXT DEFAULT CURRENT_TIMESTAMP,
                    fecha_actualizacion TEXT DEFAULT CURRENT_TIMESTAMP,
                    FOREIGN KEY (id_categoria) REFERENCES categorias(id_categoria),
                    FOREIGN KEY (id_proveedor) REFERENCES proveedores(id_proveedor)
                );
                CREATE TABLE movimientos (
                    id_movimiento INTEGER PRIMARY KEY,
                    tipo TEXT CHECK(tipo IN ('entrada', 'salida')),
                    id_producto INTEGER NOT NULL,
                    id_cliente INTEGER,
                    id_proveedor INTEGER,
                    cantidad INTEGER NOT NULL,
                    precio_unitario REAL,
                    total REAL,
                    stock_anterior INTEGER,
                    stock_posterior INTEGER,
                    referencia_documento TEXT,
                    motivo TEXT,
                    id_usuario INTEGER NOT NULL,
                    activo INTEGER DEFAULT 1,
                    fecha_movimiento TEXT DEFAULT CURRENT_TIMESTAMP,
                    FOREIGN KEY (id_producto) REFERENCES productos(id_producto),
                    FOREIGN KEY (id_cliente) REFERENCES clientes(id_cliente),
                    FOREIGN KEY (id_proveedor) REFERENCES proveedores(id_proveedor),
                    FOREIGN KEY (id_usuario) REFERENCES usuarios(id_usuario)
                );
            """)
            
            # Insertar datos de prueba (CON activo en movimientos)
            cursor.executemany("INSERT INTO categorias (id_categoria, nombre, descripcion, id_padre, nivel) VALUES (?, ?, ?, ?, ?)",
                [(1, 'Electronica', 'Dispositivos electronicos', None, 0),
                 (2, 'Computadoras', 'PCs y laptops', 1, 1),
                 (3, 'Smartphones', 'Telefonos inteligentes', 1, 1),
                 (4, 'Alimentos', 'Productos comestibles', None, 0),
                 (5, 'Bebidas', 'Bebidas', 4, 1),
                 (6, 'Textil', 'Ropa y accesorios', None, 0)])
            
            cursor.executemany("INSERT INTO proveedores (id_proveedor, razon_social, nombre_comercial, rfc, telefono, email, contacto_nombre, dias_credito) VALUES (?, ?, ?, ?, ?, ?, ?, ?)",
                [(1, 'TechSupply S.A.', 'TechSupply', 'TSC123456ABC', '5550101234', 'ventas@tech.com', 'Juan Perez', 30),
                 (2, 'Alimentos del Norte', 'ADNorte', 'ADN987654XYZ', '5550202567', 'pedidos@adn.com', 'Maria Lopez', 15),
                 (3, 'Textiles Mexicanos', 'TexMex', 'TXM456789DEF', '5550303890', 'contacto@txm.com', 'Carlos Ruiz', 0)])
            
            cursor.executemany("INSERT INTO clientes (id_cliente, tipo_cliente, nombre, apellidos, rfc, email, telefono, direccion, limite_credito) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)",
                [(1, 'persona', 'Juan', 'Perez Garcia', 'PEGJ800101HDF', 'juan@email.com', '5551001456', 'Calle 1', 5000),
                 (2, 'empresa', 'Soluciones Tecnologicas del Sur', None, 'STS900202ABC', 'sts@empresa.com', '5552002789', 'Av. Empresarial 100', 50000),
                 (3, 'persona', 'Maria', 'Lopez Santos', 'LOSM850303XYZ', 'maria@email.com', '5551003234', 'Calle 3', 10000)])
            
            cursor.executemany("INSERT INTO usuarios (id_usuario, username, password_hash, nombre_completo, email, rol) VALUES (?, ?, ?, ?, ?, ?)",
                [(1, 'admin', '[HASH_SEGURO]', 'Administrador del Sistema', 'admin@empresa.com', 'admin'),
                 (2, 'jperez', '[HASH_SEGURO]', 'Juan Perez Lopez', 'jperez@empresa.com', 'vendedor'),
                 (3, 'mgarcia', '[HASH_SEGURO]', 'Maria Garcia Ruiz', 'mgarcia@empresa.com', 'almacenista')])
            
            cursor.executemany("INSERT INTO productos (id_producto, sku, nombre, descripcion, id_categoria, id_proveedor, precio_costo, precio_venta, stock_actual, stock_minimo, ubicacion_almacen) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)",
                [(1, 'LAP-001', 'Laptop Pro 15"', 'Laptop alta performance', 2, 1, 8500, 12000, 25, 5, 'A-01-05'),
                 (2, 'TEL-001', 'Smartphone X 128GB', 'Telefono ultima generacion', 3, 1, 5000, 8000, 15, 3, 'A-02-03'),
                 (3, 'ARR-001', 'Arroz Integral 1kg', 'Arroz de grano largo', 4, 2, 18.50, 28, 150, 20, 'B-01-01'),
                 (4, 'CAM-001', 'Camiseta Algodon M', 'Camiseta basica manga corta', 6, 3, 85, 199, 50, 10, 'C-01-01'),
                 (5, 'LAP-002', 'Laptop Gamer 17"', 'Laptop gaming RTX 4060', 2, 1, 15000, 22000, 8, 2, 'A-01-06')])
            
            # Datos de prueba para movimientos (CON activo)
            cursor.executemany("INSERT INTO movimientos (id_movimiento, tipo, id_producto, id_cliente, id_proveedor, cantidad, precio_unitario, total, stock_anterior, stock_posterior, id_usuario, referencia_documento, motivo, activo) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)",
                [(1, 'entrada', 1, None, 1, 50, 8500, 425000, 0, 50, 1, 'FAC-001', 'Compra inicial', 1),
                 (2, 'entrada', 2, None, 1, 30, 5000, 150000, 0, 30, 1, 'FAC-002', 'Compra inicial', 1),
                 (3, 'entrada', 3, None, 2, 200, 18.50, 3700, 0, 200, 1, 'FAC-003', 'Compra inicial', 1),
                 (4, 'salida', 1, 1, None, 5, 12000, 60000, 50, 45, 2, 'REM-001', 'Venta a cliente', 1),
                 (5, 'salida', 2, 2, None, 3, 8000, 24000, 30, 27, 2, 'REM-002', 'Venta empresa', 1)])
            
            conn.commit()
            
            # REFRESCAR TODOS LOS DROPDOWNS DESPUES DEL RESET
            print("[SISTEMA] Datos reseteados. Actualizando todos los dropdowns...")
            for nombre, crud_data in crud_registry.items():
                if crud_data['refrescar']:
                    crud_data['refrescar']()
            
            with self.output_resultado:
                clear_output()
                display(widgets.HTML(
                    "<div style='color: #155724; background: #d4edda; padding: 10px; "
                    "border-radius: 5px;'><b>Datos de prueba restaurados correctamente</b><br>"
                    "Todas las tablas han sido recreadas con los datos originales.<br>"
                    "Todos los dropdowns han sido sincronizados.</div>"
                ))
                
        except Exception as e:
            self.mostrar_error("Error al resetear datos: {}".format(str(e)))
    
    def mostrar(self):
        return self.vista

# Crear instancia de consola SQL
consola_sql = ConsolaSQL()

print("Consola SQL Libre configurada (con deteccion automatica de cambios)")


Consola SQL Libre configurada (con deteccion automatica de cambios)


In [8]:
# ============================================
# CELDA 8: Interfaz final completa con REFRESCO AL CAMBIAR DE PESTAÑA
# ============================================

# Crear lista de vistas CRUD
vistas_crud = [
    crud_categorias.mostrar(),
    crud_proveedores.mostrar(),
    crud_clientes.mostrar(),
    crud_usuarios.mostrar(),
    crud_productos.mostrar(),
    crud_movimientos.mostrar()
]

# Tabs para las 6 tablas
tabs_crud = widgets.Tab(children=vistas_crud)

titulos = ['Categorias', 'Proveedores', 'Clientes', 'Usuarios', 'Productos', 'Movimientos']
for i, titulo in enumerate(titulos):
    tabs_crud.set_title(i, titulo)

# ============================================
# SISTEMA DE REFRESCO AL CAMBIAR DE PESTAÑA
# ============================================

def on_cambio_pestaña(change):
    """
    Se ejecuta automaticamente cuando el usuario cambia de pestaña.
    Refresca los dropdowns de la pestaña destino para asegurar datos actualizados.
    """
    nueva_pestaña = change['new']
    nombre_tabla = titulos[nueva_pestaña].lower()
    
    print("[UI] Cambiando a pestaña: {}".format(titulos[nueva_pestaña]))
    
    # Refrescar dropdowns de la tabla seleccionada si tiene funcion de refresco
    if nombre_tabla in crud_registry:
        refrescar_func = crud_registry[nombre_tabla]['refrescar']
        if refrescar_func:
            refrescar_func()
            print("  -> Dropdowns de {} refrescados automaticamente".format(titulos[nueva_pestaña]))

# Conectar el observer al evento de cambio de pestaña
tabs_crud.observe(on_cambio_pestaña, names='selected_index')

# Header
header = widgets.HTML("""
    <div style='background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); 
                color: white; padding: 20px; border-radius: 10px; margin-bottom: 15px;
                box-shadow: 0 4px 6px rgba(0,0,0,0.1); font-family: sans-serif;'>
        <h1 style='margin: 0; font-size: 28px;'>Sistema de Inventario CRUD + SQL</h1>
        <p style='margin: 8px 0 0 0; opacity: 0.9; font-size: 14px;'>
            Interfaz grafica flexible con validaciones + Consola SQL Libre | JupyterLite + SQLite
        </p>
        <p style='margin: 5px 0 0 0; opacity: 0.8; font-size: 12px; color: #d4edda;'>
            ✓ Sincronizacion automatica de dropdowns entre tablas | ✓ Refresco al cambiar de pestaña
        </p>
    </div>
""")

# Instrucciones detalladas
instrucciones = widgets.HTML("""
    <div style='background: #f8f9fa; padding: 15px; border-radius: 5px; margin-bottom: 15px; 
                border-left: 4px solid #667eea; font-family: sans-serif;'>
        <h4 style='margin-top: 0; color: #495057;'>Guia de Uso - Modo Grafico (Pestanas superiores):</h4>
        <ul style='margin: 5px 0; color: #6c757d; line-height: 1.6;'>
            <li><b>[+] Insertar:</b> Captura el ID (sugerido, pero editable), completa campos obligatorios (*) 
                y presiona <i>Guardar</i>. Valida unicidad de ID.</li>
            <li><b>[*] Modificar:</b> Escribe el ID en el campo correspondiente, presiona <i>[Cargar Datos]</i>, 
                edita los campos habilitados y presiona <i>Actualizar</i>.</li>
            <li><b>[-] Borrar:</b> Escribe el ID, presiona <i>[Cargar Datos]</i> para verificar, 
                luego <i>Confirmar Borrado</i> y confirma en el dialogo (Soft Delete).</li>
            <li><b>[?] Consultar:</b> Ingresa criterios en cualquier campo (usa * como comodin) 
                y presiona <i>Buscar</i>. Busqueda aproximada automatica.</li>
        </ul>
        <h4 style='color: #495057; margin-top: 15px;'>Sistema de Sincronizacion Automatica:</h4>
        <ul style='margin: 5px 0; color: #6c757d; line-height: 1.6;'>
            <li>Al insertar/modificar/borrar en cualquier tabla, los dropdowns relacionados se actualizan automaticamente</li>
            <li>Al cambiar de pestaña, los dropdowns se refrescan para mostrar los datos mas recientes</li>
            <li>La Consola SQL tambien detecta cambios y sincroniza las vistas graficas</li>
        </ul>
        <h4 style='color: #495057; margin-top: 15px;'>Consola SQL (Parte inferior):</h4>
        <ul style='margin: 5px 0; color: #6c757d; line-height: 1.6;'>
            <li>Ejecuta SELECT, INSERT, UPDATE, DELETE directamente</li>
            <li>Historial de comandos con flechas Arriba/Abajo</li>
            <li>Resetear Datos restaura el estado inicial y sincroniza todo</li>
        </ul>
    </div>
""")

# Separador visual
separador = widgets.HTML("""
    <div style='border-top: 3px dashed #dee2e6; margin: 20px 0;'></div>
""")

# Ensamblar aplicacion completa
app_completa = widgets.VBox([
    header,
    instrucciones,
    tabs_crud,
    separador,
    consola_sql.mostrar()
], layout=widgets.Layout(padding='10px', max_width='1200px', margin='0 auto'))

display(app_completa)

print("=" * 70)
print("SISTEMA COMPLETO CARGADO - VERSION CON SINCRONIZACION AUTOMATICA")
print("=" * 70)
print("CARACTERISTICAS PRINCIPALES:")
print("   ✓ IDs manuales con sugerencia automatica (max+1)")
print("   ✓ Validaciones basicas: numeros, rangos, unicidad de ID")
print("   ✓ Sin validaciones de formato (RFC, email, telefono)")
print("   ✓ Modificar: Carga por ID, edicion controlada")
print("   ✓ Borrar: Soft delete con confirmacion visual")
print("   ✓ Consultar: Busqueda aproximada + metacaracteres (*)")
print("   ✓ 6 tablas completas con campo activo en todas")
print("   ✓ 5 movimientos de prueba pre-cargados")
print("   ✓ SINCRONIZACION AUTOMATICA DE DROPDOWNS ENTRE TABLAS")
print("   ✓ Refresco automatico al cambiar de pestaña")
print("=" * 70)

SISTEMA COMPLETO CARGADO - VERSION CON SINCRONIZACION AUTOMATICA
CARACTERISTICAS PRINCIPALES:
   ✓ IDs manuales con sugerencia automatica (max+1)
   ✓ Validaciones basicas: numeros, rangos, unicidad de ID
   ✓ Sin validaciones de formato (RFC, email, telefono)
   ✓ Modificar: Carga por ID, edicion controlada
   ✓ Borrar: Soft delete con confirmacion visual
   ✓ Consultar: Busqueda aproximada + metacaracteres (*)
   ✓ 6 tablas completas con campo activo en todas
   ✓ 5 movimientos de prueba pre-cargados
   ✓ SINCRONIZACION AUTOMATICA DE DROPDOWNS ENTRE TABLAS
   ✓ Refresco automatico al cambiar de pestaña
